# Partidos descargados de github (2000-2024)

In [ ]:
# Unificacion en un dataframe
import pandas as pd
import os
import glob

# --- 1. Definir la ruta (¡DEBES MODIFICAR ESTO!) ---

# Ruta a la CARPETA que contiene los ~40 archivos (2000-2024)
RUTA_CARPETA_2000_2024 = r"/Users/UJA/Desktop/Datos/Excel con todos los partidos"

print("Iniciando la carga de datos (Paso 1: 2000-2024)...")

# --- 2. Cargar y Fusionar los datos 2000-2024 ---
print(f"Buscando archivos .csv en: {RUTA_CARPETA_2000_2024}")

# Encontrar todos los archivos que terminan en .csv
patron_archivos = os.path.join(RUTA_CARPETA_2000_2024, "*.csv")
lista_de_archivos_csv = glob.glob(patron_archivos)

if not lista_de_archivos_csv:
    print(f"¡Error! No se encontraron archivos .csv en la carpeta especificada.")
else:
    print(f"Se encontraron {len(lista_de_archivos_csv)} archivos .csv. Cargando...")

    lista_dfs = []

    for archivo in lista_de_archivos_csv:
        try:
            # Cargar cada archivo
            df_temporal = pd.read_csv(archivo, low_memory=False)
            lista_dfs.append(df_temporal)
        except Exception as e:
            print(f"Error al cargar el archivo {archivo}: {e}")

    # Combinar todos los DataFrames de la lista en uno solo
    df_partidos_2000_2024 = pd.concat(lista_dfs, ignore_index=True)

    print(f"\n--- ¡Éxito! Datos 2000-2024 fusionados. ---")
    print(f"Total de filas: {len(df_partidos_2000_2024)}")

    # Ordenar por fecha (¡CRUCIAL para los pipelines EWM/ELO!)
    # Asumiendo que la columna de fecha se llama 'Date' (o similar)
    if 'Date' in df_partidos_2000_2024.columns:
        df_partidos_2000_2024['Date'] = pd.to_datetime(df_partidos_2000_2024['Date'])
        df_partidos_2000_2024 = df_partidos_2000_2024.sort_values(by='Date').reset_index(drop=True)
        print(f"DataFrame ordenado por 'Date'.")
    else:
        print("Advertencia: No se encontró la columna 'Date'. El DataFrame no está ordenado cronológicamente.")

    display(df_partidos_2000_2024.head())

Iniciando la carga de datos (Paso 1: 2000-2024)...
Buscando archivos .csv en: /Users/UJA/Desktop/Datos/Excel con todos los partidos
Se encontraron 50 archivos .csv. Cargando...

--- ¡Éxito! Datos 2000-2024 fusionados. ---
Total de filas: 248895
Advertencia: No se encontró la columna 'Date'. El DataFrame no está ordenado cronológicamente.


,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2000-301,Auckland,Hard,32,A,20000110,1,103163,1.0,NaN,...,55.0,39.0,29.0,17.0,4.0,7.0,11.0,1612.0,63.0,595.0
1,2000-301,Auckland,Hard,32,A,20000110,2,102607,NaN,Q,...,32.0,25.0,18.0,12.0,3.0,6.0,211.0,157.0,49.0,723.0
2,2000-301,Auckland,Hard,32,A,20000110,3,103252,NaN,NaN,...,33.0,20.0,7.0,8.0,7.0,11.0,48.0,726.0,59.0,649.0
3,2000-301,Auckland,Hard,32,A,20000110,4,103507,7.0,NaN,...,43.0,29.0,14.0,10.0,6.0,8.0,45.0,768.0,61.0,616.0
4,2000-301,Auckland,Hard,32,A,20000110,5,102103,NaN,Q,...,46.0,34.0,18.0,12.0,5.0,9.0,167.0,219.0,34.0,873.0


In [ ]:
# Adaptacion del dataframe al formato optimo

import pandas as pd
import numpy as np

# (Asumimos que se llama 'df_partidos_2000_2024')
df = df_partidos_2000_2024.copy()

print("Iniciando la adaptación del formato 'winner/loser' a 'player_A/player_B'...")

# --- 1. Aleatorización A/B ---
swap_mask = np.random.rand(len(df)) > 0.5
df['playerA_won'] = (~swap_mask).astype(int)

# --- 2. Mapeo de Columnas de Jugador (A/B) ---
df['player_A'] = np.where(swap_mask, df['loser_name'], df['winner_name'])
df['player_B'] = np.where(swap_mask, df['winner_name'], df['loser_name'])
df['ht_A'] = np.where(swap_mask, df['loser_ht'], df['winner_ht'])
df['ht_B'] = np.where(swap_mask, df['winner_ht'], df['loser_ht'])
df['age_A'] = np.where(swap_mask, df['loser_age'], df['winner_age'])
df['age_B'] = np.where(swap_mask, df['winner_age'], df['loser_age'])
df['hand_A_encoded'] = np.where(swap_mask, df['loser_hand'], df['winner_hand'])
df['hand_B_encoded'] = np.where(swap_mask, df['winner_hand'], df['loser_hand'])
df['Rk'] = np.where(swap_mask, df['loser_rank'], df['winner_rank'])
df['vRk'] = np.where(swap_mask, df['winner_rank'], df['loser_rank'])

# --- 3.5: Rellenar NaNs de Estadísticas en Origen ---
print("Rellenando NaNs en las columnas de estadísticas en bruto...")
stat_columns_to_fill = [
    'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon', 'w_bpSaved', 'w_bpFaced', 'w_SvGms',
    'l_ace', 'l_df', 'l_svpt', 'l_1stIn', 'l_1stWon', 'l_2ndWon', 'l_bpSaved', 'l_bpFaced', 'l_SvGms'
]
existing_stat_cols = [col for col in stat_columns_to_fill if col in df.columns]
df[existing_stat_cols] = df[existing_stat_cols].fillna(0)

# --- 4. Mapeo de Estadísticas de Partido (A/B) ---
# (Mapeo básico de stats)
df['playerA_ace'] = np.where(swap_mask, df['l_ace'], df['w_ace'])
df['playerB_ace'] = np.where(swap_mask, df['w_ace'], df['l_ace'])
df['playerA_df'] = np.where(swap_mask, df['l_df'], df['w_df'])
df['playerB_df'] = np.where(swap_mask, df['w_df'], df['l_df'])
df['playerA_svpt'] = np.where(swap_mask, df['l_svpt'], df['w_svpt'])
df['playerB_svpt'] = np.where(swap_mask, df['w_svpt'], df['l_svpt'])
df['playerA_1stIn'] = np.where(swap_mask, df['l_1stIn'], df['w_1stIn'])
df['playerB_1stIn'] = np.where(swap_mask, df['w_1stIn'], df['l_1stIn'])
df['playerA_1stWon'] = np.where(swap_mask, df['l_1stWon'], df['w_1stWon'])
df['playerB_1stWon'] = np.where(swap_mask, df['w_1stWon'], df['l_1stWon'])
df['playerA_2ndWon'] = np.where(swap_mask, df['l_2ndWon'], df['w_2ndWon'])
df['playerB_2ndWon'] = np.where(swap_mask, df['w_2ndWon'], df['l_2ndWon'])
df['playerA_bpFaced'] = np.where(swap_mask, df['l_bpFaced'], df['w_bpFaced'])
df['playerB_bpFaced'] = np.where(swap_mask, df['w_bpFaced'], df['l_bpFaced'])
df['playerA_bpSaved'] = np.where(swap_mask, df['l_bpSaved'], df['w_bpSaved'])
df['playerB_bpSaved'] = np.where(swap_mask, df['w_bpSaved'], df['l_bpSaved'])

# --- 4.1 Cálculo de Estadísticas (Seguro) ---
df['playerA_DR'] = np.where(df['playerB_ace'] == 0, df['playerA_ace'], df['playerA_ace'] / df['playerB_ace'])
df['playerB_DR'] = np.where(df['playerA_ace'] == 0, df['playerB_ace'], df['playerB_ace'] / df['playerA_ace'])
df['playerA_A%'] = np.where(df['playerA_svpt'] == 0, 0, df['playerA_ace'] / df['playerA_svpt'])
df['playerB_A%'] = np.where(df['playerB_svpt'] == 0, 0, df['playerB_ace'] / df['playerB_svpt'])
df['playerA_DF%'] = np.where(df['playerA_svpt'] == 0, 0, df['playerA_df'] / df['playerA_svpt'])
df['playerB_DF%'] = np.where(df['playerB_svpt'] == 0, 0, df['playerB_df'] / df['playerB_svpt'])
df['playerA_1st%'] = np.where(df['playerA_svpt'] == 0, 0, df['playerA_1stIn'] / df['playerA_svpt'])
df['playerB_1st%'] = np.where(df['playerB_svpt'] == 0, 0, df['playerB_1stIn'] / df['playerB_svpt'])
denom_A_2nd = df['playerA_svpt'] - df['playerA_1stIn']
denom_B_2nd = df['playerB_svpt'] - df['playerB_1stIn']
df['playerA_2nd%'] = np.where(denom_A_2nd <= 0, 0, df['playerA_2ndWon'] / denom_A_2nd)
df['playerB_2nd%'] = np.where(denom_B_2nd <= 0, 0, df['playerB_2ndWon'] / denom_B_2nd)
df['playerA_BPSvd%'] = np.where(df['playerA_bpFaced'] == 0, 0, df['playerA_bpSaved'] / df['playerA_bpFaced'])
df['playerB_BPSvd%'] = np.where(df['playerB_bpFaced'] == 0, 0, df['playerB_bpSaved'] / df['playerB_bpFaced'])

df['Total_juegos'] = df['w_SvGms'] + df['l_SvGms']
df['Numero_sets'] = df['score'].str.count('-') + 1

# --- 5. Renombrar y Codificar Columnas de Contexto ---
print("Codificando columnas de contexto (Surface, Tier, Round)...")
df.rename(columns={
    'tourney_date': 'Date', # <- La columna 'Date' ahora tiene los enteros
    'tourney_name': 'Tournament',
    'best_of': 'Best_of_5',
    'round': 'Round',
    'minutes': 'duracion_min',
    'score': 'Score'
}, inplace=True)

surface_map = {'Hard': 2, 'Clay': 3, 'Grass': 1, 'Carpet': 4}
df['Surface_Encoded'] = df['surface'].map(surface_map).fillna(0).astype(int)

tier_map = {'G': 5, 'M': 4, 'F': 3, 'A': 2, 'C': 1, 'D': 1}
df['Tier_Numeric'] = df['tourney_level'].map(tier_map).fillna(1).astype(int)

round_map = {'F': 7, 'SF': 6, 'QF': 5, 'R16': 4, 'R32': 3, 'R64': 2, 'R128': 1, 'RR': 0}
df['Rd_Ordinal'] = df['Round'].map(round_map).fillna(0).astype('category')

# --- 6. Limpieza Final ---
print("Limpiando columnas 'winner/loser' y 'w/l' originales...")
columnas_a_conservar = [
    'Date', 'Tournament', 'Score', 'Best_of_5', 'Round', 'duracion_min',
    'player_A', 'player_B', 'playerA_won',
    'ht_A', 'ht_B', 'age_A', 'age_B', 'hand_A_encoded', 'hand_B_encoded', 'Rk', 'vRk',
    'playerA_DR', 'playerA_A%', 'playerA_DF%', 'playerA_1st%', 'playerA_2nd%', 'playerA_BPSvd%',
    'playerB_DR', 'playerB_A%', 'playerB_DF%', 'playerB_1st%', 'playerB_2nd%', 'playerB_BPSvd%',
    'Total_juegos', 'Numero_sets',
    'Tier_Numeric', 'Surface_Encoded', 'Rd_Ordinal'
]
columnas_finales = [col for col in columnas_a_conservar if col in df.columns]
df_adaptado = df[columnas_finales]

print("--- PASO 1: Reparando 'hand_A_encoded' y 'hand_B_encoded' ---")


# --- 7. Ordenar Cronológicamente (¡CORRECCIÓN APLICADA AQUÍ!) ---
# Primero, convertimos el entero a string, LUEGO a datetime
df_adaptado['Date'] = pd.to_datetime(df_adaptado['Date'].astype(str), format='%Y%m%d')

# Ahora, el ordenamiento funcionará correctamente
df_adaptado = df_adaptado.sort_values(by='Date').reset_index(drop=True)
df_adaptado['indice'] = df_adaptado.index

# Reemplazamos el DataFrame original por el adaptado
df_partidos_2000_2024_adaptado = df_adaptado

#####este codigo comentado se deja para cuando tenga todos los datos

# # 1. Definir la función de codificación
# # (Esta función faltaba en el script de 'Adaptación' del Turno 161)
# def encode_hand(hand_series):
#     # Asegurarse de que es string, por si acaso
#     return hand_series.astype(str).str.upper().map({
#         'R': 1,  # Diestro
#         'L': -1, # Zurdo
#         'U': 0   # Desconocido/Ambi
#     }).fillna(0) # Rellenar NaNs (y 'None') con 0

# # --- 2. Arreglar df_partidos_2000_2024_adaptado ---
# print("Arreglando el dataframe 2000-2024...")
# df_partidos_2000_2024_adaptado['hand_A_encoded'] = encode_hand(df_partidos_2000_2024_adaptado['hand_A_encoded'])
# df_partidos_2000_2024_adaptado['hand_B_encoded'] = encode_hand(df_partidos_2000_2024_adaptado['hand_B_encoded'])

# # Ahora SÍ podemos calcular los _diff (Paso 1 de Turno 172)
# print("Calculando columnas '_diff' para 2000-2024...")
# df_partidos_2000_2024_adaptado['ht_diff'] = df_partidos_2000_2024_adaptado['ht_A'] - df_partidos_2000_2024_adaptado['ht_B']
# df_partidos_2000_2024_adaptado['age_diff'] = df_partidos_2000_2024_adaptado['age_A'] - df_partidos_2000_2024_adaptado['age_B']
# df_partidos_2000_2024_adaptado['hand_diff'] = df_partidos_2000_2024_adaptado['hand_A_encoded'] - df_partidos_2000_2024_adaptado['hand_B_encoded']

print("\n--- ¡Éxito! El DataFrame (2000-2024) ha sido adaptado. ---")
print(f"Total de filas: {len(df_partidos_2000_2024_adaptado)}")
print(f"Fecha de inicio: {df_partidos_2000_2024_adaptado['Date'].min()}")
print(f"Fecha de fin: {df_partidos_2000_2024_adaptado['Date'].max()}")
display(df_partidos_2000_2024_adaptado.head())

Iniciando la adaptación del formato 'winner/loser' a 'player_A/player_B'...
Rellenando NaNs en las columnas de estadísticas en bruto...
Codificando columnas de contexto (Surface, Tier, Round)...
Limpiando columnas 'winner/loser' y 'w/l' originales...
--- PASO 1: Reparando 'hand_A_encoded' y 'hand_B_encoded' ---

--- ¡Éxito! El DataFrame (2000-2024) ha sido adaptado. ---
Total de filas: 248895
Fecha de inicio: 2000-01-03 00:00:00
Fecha de fin: 2024-12-18 00:00:00


C:\Users\UJA\AppData\Local\Temp\ipykernel_2724\461436437.py:113: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_adaptado['Date'] = pd.to_datetime(df_adaptado['Date'].astype(str), format='%Y%m%d')


,Date,Tournament,Score,Best_of_5,Round,duracion_min,player_A,player_B,playerA_won,ht_A,...,playerB_DF%,playerB_1st%,playerB_2nd%,playerB_BPSvd%,Total_juegos,Numero_sets,Tier_Numeric,Surface_Encoded,Rd_Ordinal,indice
0,2000-01-03,Doha,7-6(4) 4-6 6-4,3,R32,143.0,Ivo Heuberger,Cristiano Caratti,0,188.0,...,0.009346,0.672897,0.457143,0.642857,32.0,4.0,2,2,3.0,0
1,2000-01-03,Chennai,6-3 6-4,3,R32,81.0,Martin Spottl,Jerome Golmard,0,180.0,...,0.030769,0.661538,0.636364,1.000000,19.0,3.0,2,2,3.0,1
2,2000-01-03,Chennai,6-4 6-4,3,R32,82.0,Sunil Kumar,Ronald Agenor,0,NaN,...,0.000000,0.546875,0.655172,0.666667,20.0,3.0,2,2,3.0,2
3,2000-01-03,Chennai,6-3 6-0,3,R32,50.0,Martin Damm Sr,Christophe Rochus,1,188.0,...,0.027027,0.459459,0.450000,0.200000,15.0,3.0,2,2,3.0,3
4,2000-01-03,Chennai,7-5 6-2,3,R32,80.0,Tomas Zib,Lorenzo Manta,0,178.0,...,0.014493,0.608696,0.592593,1.000000,20.0,3.0,2,2,3.0,4


In [ ]:
df_partidos_2000_2024_adaptado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248895 entries, 0 to 248894
Data columns (total 35 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   Date             248895 non-null  datetime64[ns]
 1   Tournament       248895 non-null  object        
 2   Score            248877 non-null  object        
 3   Best_of_5        248895 non-null  int64         
 4   Round            248895 non-null  object        
 5   duracion_min     178773 non-null  float64       
 6   player_A         248895 non-null  object        
 7   player_B         248895 non-null  object        
 8   playerA_won      248895 non-null  int64         
 9   ht_A             228440 non-null  float64       
 10  ht_B             228553 non-null  float64       
 11  age_A            248721 non-null  float64       
 12  age_B            248756 non-null  float64       
 13  hand_A_encoded   248889 non-null  object        
 14  hand_B_encoded   248

In [ ]:
# Extraer la Lista de Torneos
import pandas as pd

# (Asumimos que 'df_partidos_2000_2024_adaptado' existe)

print("--- Extrayendo Lista Maestra de Torneos (2000-2024) ---")

# Creamos un dataframe con el nombre único de cada torneo y su Tier
df_torneos = df_partidos_2000_2024_adaptado[
    ['Tournament', 'Tier_Numeric']
].drop_duplicates().sort_values(by=['Tier_Numeric', 'Tournament'])

print(f"Total de torneos únicos encontrados: {len(df_torneos)}")

# (Opcional) Ver los torneos por Tier
print("\n--- Muestra de Torneos por Tier ---")
print("\nATP Tour (Tier >= 2):")
display(df_torneos[df_torneos['Tier_Numeric'] >= 2].head())

print("\nChallengers (Tier == 1):")
display(df_torneos[df_torneos['Tier_Numeric'] == 1].head())

# (Opcional) Guardar las listas si las necesitas
lista_torneos_atp = df_torneos[df_torneos['Tier_Numeric'] >= 2]['Tournament'].unique()
lista_torneos_challenger = df_torneos[df_torneos['Tier_Numeric'] == 1]['Tournament'].unique()

--- Extrayendo Lista Maestra de Torneos (2000-2024) ---
Total de torneos únicos encontrados: 2711

--- Muestra de Torneos por Tier ---

ATP Tour (Tier >= 2):


,Tournament,Tier_Numeric
194526,ATP Rio de Janeiro,2
8734,Acapulco,2
8,Adelaide,2
208185,Adelaide 1,2
208365,Adelaide 2,2



Challengers (Tier == 1):


,Tournament,Tier_Numeric
105374,ATP Challenger Tour Finals CH,1
7078,Aachen CH,1
70047,Aarhus CH,1
239013,Acapulco CH,1
141771,Agri CH,1


In [ ]:
lista_atp_original = [
    'Rio de Janeiro', 'Acapulco', 'Adelaide', 'Adelaide 1', 'Adelaide 2', 'Almaty',
    'Amersfoort', 'Amsterdam', 'Antalya', 'Antwerp', 'Astana', 'Athens Olympics',
    'Atlanta', 'Atp Cup', 'Auckland', 'Bangkok', 'Banja Luka', 'Barcelona', 'Basel',
    'Bastad', 'Beijing', 'Beijing Olympics', 'Belgrade', 'Belgrade ', 'Belgrade 2',
    'Bogota', 'Brighton', 'Brisbane', 'Bucharest', 'Budapest', 'Buenos Aires',
    'Cagliari', 'Casablanca', 'Chengdu', 'Chennai', 'Cologne 1', 'Cologne 2',
    'Copenhagen', 'Cordoba', 'Costa Do Sauipe', 'Dallas', 'Delray Beach', 'Doha',
    'Dubai', 'Dusseldorf', 'Eastbourne', 'Estoril', 'Florence', 'Geneva', 'Gijon',
    'Great Ocean Road Open', 'Gstaad', 'Halle', 'Hamburg', 'Hangzhou', 'Ho Chi Minh City',
    'Hong Kong', 'Houston', 'Indianapolis', 'Istanbul', 'Johannesburg', 'Kitzbuhel',
    'Kuala Lumpur', 'Las Vegas', 'Laver Cup', 'London', 'London Olympics',
    'Long Island', 'Los Angeles', 'Los Cabos', 'Lyon', 'Mallorca', 'Marbella',
    'Marrakech', 'Marseille', 'Melbourne', 'Memphis', 'Metz', 'Mexico City', 'Milan',
    'Montpellier', 'Moscow', 'Mumbai', 'Munich', 'Murray River Open', 'Naples',
    'New Haven', 'New York', 'Newport', 'NextGen Finals', 'Nice', 'Nottingham',
    'Nur-Sultan', 'Orlando', 'Palermo', 'Parma', 'Poertschach', 'Pune', "Queen's Club",
    'Quito', 'Rio De Janeiro', 'Rio Olympics', 'Rio de Janeiro', 'Rotterdam', 'San Diego',
    'San Jose', 'San Marino', 'Santiago', 'Sao Paulo', 'Sardinia', 'Scottsdale', 'Seoul',
    'Shanghai', 'Shenzhen', 'Singapore', 'Sofia', 'Sopot', 'St Petersburg',
    'St. Petersburg', 'St. Poelten', 'Stockholm', 'Stuttgart', 'Stuttgart Outdoor',
    'Sydney', 'Sydney Olympics', 'Tashkent', 'Tel Aviv', 'Tokyo', 'Tokyo Olympics',
    'Toulouse', 'Tour Finals', 'Umag', 'United Cup', 'Valencia', 'Vienna',
    'Vina del Mar', 'Warsaw', 'Washington', 'Winston-Salem', 'Zagreb', 'Zhuhai',
    's Hertogenbosch', 'Masters Cup', 'Next Gen Finals', 'Canada Masters',
    'Cincinnati Masters', 'Hamburg Masters', 'Indian Wells Masters', 'Madrid Masters',
    'Miami Masters', 'Monte Carlo Masters', 'Paris Masters', 'Rome Masters',
    'Shanghai Masters', 'Stuttgart Masters', 'Australian Open', 'Doha Aus Open Qualies',
    'Roland Garros', 'US Open', 'Us Open', 'Wimbledon'
]

# Fix: Convert the numpy array to a list of strings
lista_challenger_original = list(lista_torneos_challenger)

# 2. Nuevos torneos encontrados en el PDF
nuevos_torneos_atp = ['Davis Cup']
nuevos_torneos_challenger = [
    'Bengaluru CH', 'Canberra CH', 'Nouméa CH', 'Nonthaburi 1 CH',
    'Nottingham 1 CH', 'Nonthaburi 2 CH', 'Buenos Aires CH', 'Glasgow CH', 'Oeiras 1 CH'
]

# 3. Combinar y de-duplicar
lista_atp_final = sorted(list(set(lista_atp_original + nuevos_torneos_atp)))
lista_challenger_final = sorted(list(set(lista_challenger_original + nuevos_torneos_challenger)))

print(f"--- Lista ATP Final ({len(lista_atp_final)} torneos) ---")
# print(lista_atp_final) # Descomenta para ver la lista completa

print(f"\n--- Lista Challenger Final ({len(lista_challenger_final)} torneos) ---")
# print(lista_challenger_final) # Descomenta para ver la lista completa

--- Lista ATP Final (161 torneos) ---

--- Lista Challenger Final (2549 torneos) ---


# Preparacion y limpieza de datos

In [ ]:
# Cargar y filtra el dataframe de 2025

import pandas as pd
import numpy as np

# --- 1. Definir la ruta ---
RUTA_ARCHIVO_2025_ORIGINAL = r"todos_los_partidos.csv"

# --- 2. RE-CREAR Listas de Torneos ---
print("Paso 2: Re-creando listas de torneos...")

# (Asegúrate de que 'lista_atp_original' y 'lista_challenger_original' existan)
# lista_atp_original = list(lista_atp_original)
# lista_challenger_original = list(lista_challenger_original)

nuevos_torneos_atp = ['Davis Cup']
lista_atp_final_limpia = list(set(lista_atp_original + nuevos_torneos_atp))

nuevos_torneos_challenger = [
    'Bengaluru CH', 'Canberra CH', 'Nouméa CH', 'Nonthaburi 1 CH',
    'Nottingham 1 CH', 'Nonthaburi 2 CH', 'Buenos Aires CH', 'Glasgow CH', 'Oeiras 1 CH'
]
lista_challenger_final_limpia = list(set(lista_challenger_original + nuevos_torneos_challenger))

lista_torneos_permitidos = set(lista_atp_final_limpia + lista_challenger_final_limpia)
print(f"Se usarán {len(lista_torneos_permitidos)} nombres de torneos únicos para filtrar.")


# --- 3. Cargar y Filtrar ---
print(f"Paso 3: Cargando y filtrando {RUTA_ARCHIVO_2025_ORIGINAL}...")
try:
    df_2025_original = pd.read_csv(RUTA_ARCHIVO_2025_ORIGINAL, low_memory=False)

    # --- ¡INICIO DE LA CORRECCIÓN! ---
    print("Corrigiendo manualmente el guion no estándar (U+2011) en las fechas...")

    # 1. Asegurar que la columna 'Date' sea de tipo string
    df_2025_original['Date'] = df_2025_original['Date'].astype(str)

    # 2. Reemplazar el guion "malo" (‑) por el guion "bueno" (-)
    # (El primer guion lo he copiado de tu mensaje de error; el segundo es del teclado)
    df_2025_original['Date'] = df_2025_original['Date'].str.replace('‑', '-')

    # 3. Ahora que el string está limpio, lo convertimos usando el formato estándar
    df_2025_original['Date'] = pd.to_datetime(
        df_2025_original['Date'], format='%d-%b-%Y'
    )
    # --- ¡FIN DE LA CORRECCIÓN! ---

    # Filtrar por Fecha Y por Torneo
    mask_fecha_2025 = (df_2025_original['Date'] >= '2025-01-01') & \
                      (df_2025_original['Date'] < '2026-01-01')

    mask_torneos_permitidos = df_2025_original['Tournament'].isin(lista_torneos_permitidos)

    df_2025_filtrado = df_2025_original[mask_fecha_2025 & mask_torneos_permitidos].copy()

    print(f"\n--- ¡Éxito! Datos de 2025 Cargados y Filtrados ---")
    print(f"Filas originales en 2025 (basado en fecha): {len(df_2025_original[mask_fecha_2025])}")
    print(f"Filas filtradas (limpias): {len(df_2025_filtrado)}")

except FileNotFoundError:
    print(f"¡Error! No se encontró el archivo de 2025 en: {RUTA_ARCHIVO_2025_ORIGINAL}")
except Exception as e:
    print(f"Un error ocurrió al cargar los datos de 2025: {e}")

Paso 2: Re-creando listas de torneos...
Se usarán 2710 nombres de torneos únicos para filtrar.
Paso 3: Cargando y filtrando /Users/UJA/Desktop/Datos/todos_los_partidos.csv...
Corrigiendo manualmente el guion no estándar (U+2011) en las fechas...

--- ¡Éxito! Datos de 2025 Cargados y Filtrados ---
Filas originales en 2025 (basado en fecha): 55598
Filas filtradas (limpias): 22098


In [ ]:
import pandas as pd
import numpy as np
df_partidos = df_2025_filtrado
df_partidos


,Date,Tournament,Surface,Rd,Rk,vRk,Unnamed: 6,Score,More,DR,A%,DF%,1stIn,1st%,2nd%,BPSvd,Time,jugador
1,2025-10-27,Paris Masters,Hard,R32,1.0,31.0,Cameron Norrie[GBR]d.(1)Alcaraz,4-6 6-3 6-4,(ch),0.77,8.4%,2.1%,61.1%,63.8%,59.5%,5/7,2:22,Carlos Alcaraz
2,2025-09-24,Tokyo,Hard,F,1.0,5.0,(1)Alcarazd.(2)Taylor Fritz[USA],6-4 6-4,(ch),1.21,9.5%,4.8%,63.5%,77.5%,43.5%,2/3,1:33,Carlos Alcaraz
3,2025-09-24,Tokyo,Hard,SF,1.0,12.0,(1)Alcarazd.(4)Casper Ruud[NOR],3-6 6-3 6-4,(ch),1.15,13.8%,2.5%,63.7%,78.4%,55.2%,3/4,2:08,Carlos Alcaraz
4,2025-09-24,Tokyo,Hard,QF,1.0,33.0,(1)Alcarazd.Brandon Nakashima[USA],6-2 6-4,(ch),2.37,2.2%,2.2%,66.7%,83.3%,80.0%,0/0,1:20,Carlos Alcaraz
5,2025-09-24,Tokyo,Hard,R16,1.0,45.0,(1)Alcarazd.Zizou Bergs[BEL],6-4 6-3,(ch),1.15,5.0%,3.3%,61.7%,67.6%,39.1%,3/6,1:19,Carlos Alcaraz
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390496,2025-01-06,Nonthaburi 2 CH,Hard,R32,429.0,775.0,(WC)Maximus Jones[THA]d.(Q)Paire,6-7(3) 6-3 6-4,NaN,0.84,3.2%,7.4%,59.6%,69.6%,57.9%,6/11,1:57,Jin Peng Tang
390497,2025-01-06,Nonthaburi 2 CH,Hard,Q2,429.0,629.0,(4)Paired.(Alt)Mitsuki Wei Kang Leong[MAS],6-4 6-3,NaN,1.36,7.0%,3.5%,63.2%,75.0%,52.4%,3/4,1:16,Jin Peng Tang
390498,2025-01-06,Nonthaburi 2 CH,Hard,Q1,429.0,1454.0,(4)Paired.(WC)Credit Chaiyarin[THA],6-3 7-6(5),NaN,1.31,14.5%,8.7%,55.1%,84.2%,45.2%,2/4,1:23,Jin Peng Tang
392202,2025-10-06,Cali CH,Clay,Q1,1903.0,384.0,(2)Guido Ivan Justo[ARG]d.(WC)Robledo Hoyos,6-2 6-0,NaN,0.41,0.0%,7.0%,53.5%,60.9%,15.0%,3/8,1:01,Pablo Robledo Hoyos


In [ ]:
#cambio el tipo de "Date"
df_partidos["Date"] = df_partidos["Date"].astype("str")
df_partidos["Date"] = pd.to_datetime(df_partidos['Date'].str.replace('‑', '-'), format='%Y-%m-%d', errors='coerce')
df_partidos.sort_values(by="Date")

,Date,Tournament,Surface,Rd,Rk,vRk,Unnamed: 6,Score,More,DR,A%,DF%,1stIn,1st%,2nd%,BPSvd,Time,jugador
11367,2025-01-06,Auckland,Hard,R32,49.0,184.0,Mensikd.(LL)Pablo Carreno Busta[ESP],6-2 4-6 7-5,NaN,1.34,20.5%,2.3%,62.5%,81.8%,54.5%,1/2,2:11,Jakub Mensik
51794,2025-01-06,Adelaide,Hard,R32,173.0,39.0,(8)Tomas Martin Etcheverry[ARG]d.(WC)Schoolkate,7-6(8) 6-3,NaN,0.83,21.5%,1.5%,55.4%,86.1%,44.8%,1/2,1:51,Tristan Schoolkate
306897,2025-01-06,Oeiras 1 CH,Hard,Q1,1340.0,325.0,(5)Alexey Zakharov[RUS]d.(WC)Luis,6-1 3-6 6-2,NaN,0.62,0.0%,5.6%,60.6%,58.1%,39.3%,3/8,1:37,Tomas Luis
85163,2025-01-06,Nottingham CH,Hard,R32,381.0,404.0,Mikrutd.Jonas Forejtek[CZE],6-7(2) 6-3 6-1,NaN,1.32,18.5%,3.7%,69.1%,78.6%,48.0%,0/1,1:50,Luka Mikrut
241869,2025-01-06,Nottingham CH,Hard,QF,461.0,433.0,(WC)Matusevichd.(LL)Norbert Gombos[SVK],4-6 6-4 6-3,NaN,1.11,12.5%,0.0%,64.8%,82.5%,41.9%,0/2,1:40,Anton Matusevich
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109607,2025-11-10,Kobe CH,Hard,R32,220.0,249.0,Hsud.Yuta Shimizu[JPN],7-6(5) 6-4,NaN,1.12,8.2%,0.0%,60.3%,68.2%,41.4%,1/4,1:44,Yu Hsiou Hsu
58020,2025-11-10,Montevideo CH,Clay,R32,107.0,270.0,(6)BurruchagavsFederico Agustin Gomez[ARG],Live Scores,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Roman Andres Burruchaga
339588,2025-11-10,Champaign CH,Hard,Q1,1113.0,1225.0,Noah Zamora[USA]d.(12)Perez,7-6(2) 6-1,NaN,0.68,4.5%,3.4%,59.1%,59.6%,33.3%,5/11,1:29,Brandon Perez
60311,2025-11-10,Montevideo CH,Clay,R32,111.0,228.0,(8)Barrios Veravs(NG)Juan Carlos Prado Angelo[...,Live Scores,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Tomas Barrios Vera


In [ ]:
#Selecciono solo los años 2000 para adelante

df_partidos = df_partidos[df_partidos["Date"].dt.year >= 2000]
df_partidos.Date.dt.year.unique()

#Elimino columnas no necesarias

df_partidos = df_partidos.drop(columns=[ 'More'])

In [ ]:
#Elimino torneos especiales y diferentes (juegos olimipicos, ATP cup y Davis Cup)

# Partidos Davis Cup

# Cuenta cuántos partidos de la Copa Davis hay antes de eliminar
davis_cup_matches = df_partidos[df_partidos['Tournament'].str.contains("Davis Cup")].shape[0]

# Filtra el DataFrame para MANTENER solo los partidos que NO (~) contienen "Davis Cup"
# El argumento na=False es importante si tienes valores nulos en esa columna
df_partidos = df_partidos[~df_partidos['Tournament'].str.contains("Davis Cup", na=False)]

# Partidos Juegos olimpicos

# Cuenta cuántos partidos de la Juegos olimpicos hay antes de eliminar
juegos_olimpicos_matches = df_partidos[df_partidos['Tournament'].str.contains("Olympics")].shape[0]
print(f"Partidos Davis Cup: {juegos_olimpicos_matches}")
# Filtra el DataFrame para MANTENER solo los partidos que NO (~) contienen ""Olympics""
# El argumento na=False es importante si tienes valores nulos en esa columna
df_partidos = df_partidos[~df_partidos['Tournament'].str.contains("Olympics", na=False)]


#Elimino los partidos correspondientes a los torneos 'Laver Cup' y  'Atp Cup'

# --- Limpieza de Torneos No Estándar ---

# 1. Define tu lista de patrones de torneos a excluir.
#    Cualquier partido que contenga estas subcadenas será eliminado.
torneos_a_excluir = [
    'Laver Cup',
    'Atp Cup',
    "United Cup"
    # Puedes añadir aquí 'Exhibition' o cualquier otro patrón
]

# 2. Crea el patrón regex uniendo la lista con '|' (el operador 'O' en regex)
#    Esto buscará: 'Davis Cup|Laver Cup|Atp Cup'
patron_regex = '|'.join(torneos_a_excluir)

print(f"Patrón de exclusión a utilizar: {patron_regex}")

# 3. Muestra cuántos partidos se eliminarán (verificación)
partidos_a_eliminar = df_partidos[df_partidos['Tournament'].str.contains(patron_regex, na=False, regex=True)].shape[0]
print(f"Se encontraron {partidos_a_eliminar} partidos de equipo/exhibición para eliminar.")

# 4. Aplica el filtro unificado
#    Mantenemos las filas donde el nombre del torneo NO (~) contiene el patrón
df_partidos = df_partidos[~df_partidos['Tournament'].str.contains(patron_regex, na=False, regex=True)]



Partidos Davis Cup: 0
Patrón de exclusión a utilizar: Laver Cup|Atp Cup|United Cup
Se encontraron 18 partidos de equipo/exhibición para eliminar.


In [ ]:
#Creo una variable indice

# Primero, cambio el tipo y defino un orden en "Rd"

df_partidos["Rd"] = df_partidos["Rd"].astype("category")
df_partidos["Rd"] = df_partidos["Rd"].cat.set_categories(["Q1", "Q2", "Q3","Q4",'R128','R64','R32','R16','QF',"BR","RR",'SF','F'])

# Segundo, ordeno los partidos por "Rd" y "Date"

df_partidos = df_partidos.sort_values(by=["Date","Rd"])
df_partidos["indice"] = range(1,len(df_partidos)+1)

#Ordeno los partidos por "indice"

df_partidos = df_partidos.sort_values(by="indice")

In [ ]:
df_partidos.query("jugador == 'Carlos Alcaraz'")

,Date,Tournament,Surface,Rd,Rk,vRk,Unnamed: 6,Score,DR,A%,DF%,1stIn,1st%,2nd%,BPSvd,Time,jugador,indice
75,2025-01-13,Australian Open,Hard,R128,3.0,77.0,(3)Alcarazd.Alexander Shevchenko[KAZ],6-1 7-5 6-1,1.65,7.4%,4.9%,59.3%,68.8%,66.7%,2/4,1:54,Carlos Alcaraz,754
74,2025-01-13,Australian Open,Hard,R64,3.0,65.0,(3)Alcarazd.Yoshihito Nishioka[JPN],6-0 6-1 6-4,2.77,23.3%,5.0%,60.0%,88.9%,70.8%,0/0,1:21,Carlos Alcaraz,882
73,2025-01-13,Australian Open,Hard,R32,3.0,33.0,(3)Alcarazd.Nuno Borges[POR],6-2 6-4 6-7(3) 6-2,1.48,8.3%,2.8%,61.5%,83.6%,57.1%,3/3,2:55,Carlos Alcaraz,946
72,2025-01-13,Australian Open,Hard,R16,3.0,18.0,(3)Alcarazd.(15)Jack Draper[GBR],7-5 6-1 0-0 RET,1.55,10.2%,8.5%,67.8%,82.5%,36.8%,2/3,1:35,Carlos Alcaraz,1042
71,2025-01-13,Australian Open,Hard,QF,3.0,7.0,(7)Novak Djokovic[SRB]d.(3)Alcaraz,4-6 6-4 6-3 6-4,0.94,8.0%,4.0%,73.6%,67.4%,33.3%,7/13,3:37,Carlos Alcaraz,1090
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5,2025-09-24,Tokyo,Hard,R16,1.0,45.0,(1)Alcarazd.Zizou Bergs[BEL],6-4 6-3,1.15,5.0%,3.3%,61.7%,67.6%,39.1%,3/6,1:19,Carlos Alcaraz,18501
4,2025-09-24,Tokyo,Hard,QF,1.0,33.0,(1)Alcarazd.Brandon Nakashima[USA],6-2 6-4,2.37,2.2%,2.2%,66.7%,83.3%,80.0%,0/0,1:20,Carlos Alcaraz,18517
3,2025-09-24,Tokyo,Hard,SF,1.0,12.0,(1)Alcarazd.(4)Casper Ruud[NOR],3-6 6-3 6-4,1.15,13.8%,2.5%,63.7%,78.4%,55.2%,3/4,2:08,Carlos Alcaraz,18525
2,2025-09-24,Tokyo,Hard,F,1.0,5.0,(1)Alcarazd.(2)Taylor Fritz[USA],6-4 6-4,1.21,9.5%,4.8%,63.5%,77.5%,43.5%,2/3,1:33,Carlos Alcaraz,18529


In [ ]:
#Elimino partidos duplicados

import pandas as pd
import numpy as np
import re

# --- 1. CONFIGURACIÓN ---

# Las columnas estadísticas que quieres desdoblar para cada jugador
columnas_stats = ['DR', 'A%', 'DF%', '1stIn', '1st%', '2nd%', 'BPSvd']

# Definimos las columnas base (sin 'Time' ni 'Score' original para evitar fallos)
# Usaremos la huella digital más abajo.
columnas_clave_base = ['Date', 'Tournament', 'Rd']

# --- 2. FUNCIONES AUXILIARES (ROBUSTEZ) ---

def crear_huella_score(score):
    """Crea una huella única ordenando los números del marcador."""
    if pd.isna(score): return "nan"
    s = str(score)
    numeros = re.findall(r'\d+', s)
    if not numeros: return "0-0"
    numeros.sort() # Ordenar para que 6-4 y 4-6 sean iguales
    return "-".join(numeros)

def combinar_partidos_y_crear_stats(group):
    """
    Recibe un grupo (idealmente de 2 filas).
    Retorna una sola fila con playerA_stats y playerB_stats.
    """
    # Validación estricta: Si no hay 2 mitades, el partido está incompleto
    if len(group) != 2:
        return pd.DataFrame()

    # Tomamos las dos filas arbitrariamente (0 y 1)
    row_0 = group.iloc[0]
    row_1 = group.iloc[1]

    # Creamos la fila base copiando la primera fila
    new_row = row_0.copy()

    # --- ASIGNACIÓN DE JUGADORES ---
    new_row['player_A'] = row_0['jugador']
    new_row['player_B'] = row_1['jugador']

    # --- ASIGNACIÓN DE ESTADÍSTICAS (Lado a Lado) ---

    # Para el Jugador A (viene de row_0)
    for col in columnas_stats:
        # Usamos .get() para evitar errores si alguna columna no existe
        new_row[f'playerA_{col}'] = row_0.get(col, np.nan)

    # Para el Jugador B (viene de row_1)
    for col in columnas_stats:
        new_row[f'playerB_{col}'] = row_1.get(col, np.nan)

    # Retornamos como DataFrame de una fila
    return pd.DataFrame([new_row])

# --- 3. EJECUCIÓN DEL PROCESO ---

# A) Limpieza previa vital (Rescate de NaNs y Huella)
print("Preparando datos para unificación...")
df_partidos['Rd'] = df_partidos['Rd'].astype(str).replace('nan', 'Unknown')
df_partidos['score_fingerprint'] = df_partidos['Score'].apply(crear_huella_score)

# B) Definir claves finales robustas
claves_agrupamiento = ['Date', 'Tournament', 'Rd', 'score_fingerprint']

print(f"Agrupando y creando columnas para: {columnas_stats}")

# C) Aplicar la función combinada
df_unificado = df_partidos.groupby(claves_agrupamiento, as_index=False).apply(combinar_partidos_y_crear_stats)

# D) Limpieza Final
# Eliminamos las columnas auxiliares y las originales que ya están duplicadas en A/B
cols_a_borrar = ['jugador', 'score_fingerprint'] + columnas_stats
df_final = df_unificado.drop(columns=cols_a_borrar, errors='ignore')

# --- 4. CREACIÓN DEL TARGET (VARIABLE OBJETIVO) ---

# IMPORTANTE: Como asignamos A y B arbitrariamente (row 0 y row 1),
# necesitamos definir quién ganó realmente.
# Asumiendo que tienes una columna 'Winner' que indica el nombre del ganador:
if 'Winner' in df_final.columns:
    df_final['playerA_won'] = (df_final['player_A'] == df_final['Winner']).astype(int)
    print("Variable objetivo 'playerA_won' creada exitosamente.")
else:
    print("ADVERTENCIA: No se encontró columna 'Winner'. Recuerda crear 'playerA_won' manualmente.")

# --- VERIFICACIÓN ---
print("-" * 30)
print(f"Total partidos finales: {len(df_final)}")
print("Ejemplo de columnas generadas:")
print(df_final[['player_A', 'player_B', 'playerA_DR', 'playerB_DR', 'playerA_1stIn', 'playerB_1stIn']].head(2))
print("-" * 30)

df_partidos = df_final.copy()

Preparando datos para unificación...
Agrupando y creando columnas para: ['DR', 'A%', 'DF%', '1stIn', '1st%', '2nd%', 'BPSvd']
ADVERTENCIA: No se encontró columna 'Winner'. Recuerda crear 'playerA_won' manualmente.
------------------------------
Total partidos finales: 9003
Ejemplo de columnas generadas:
                      player_A         player_B playerA_DR playerB_DR  \
0 4968   Felix Auger Aliassime  Sebastian Korda       1.73       0.58   
1 69600         Rinky Hijikata      Omar Jasika       4.49       0.22   

        playerA_1stIn playerB_1stIn  
0 4968          56.9%         61.7%  
1 69600         68.4%         41.7%  
------------------------------


C:\Users\UJA\AppData\Local\Temp\ipykernel_2724\3899412110.py:74: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_unificado = df_partidos.groupby(claves_agrupamiento, as_index=False).apply(combinar_partidos_y_crear_stats)


In [ ]:
df_partidos.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 9003 entries, (np.int64(0), np.int64(4968)) to (np.int64(10041), np.int64(112123))
Data columns (total 26 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           9003 non-null   datetime64[ns]
 1   Tournament     9003 non-null   object        
 2   Surface        9003 non-null   object        
 3   Rd             9003 non-null   object        
 4   Rk             8987 non-null   float64       
 5   vRk            8942 non-null   float64       
 6   Unnamed: 6     9003 non-null   object        
 7   Score          9003 non-null   object        
 8   Time           8766 non-null   object        
 9   indice         9003 non-null   float64       
 10  player_A       9003 non-null   object        
 11  player_B       9003 non-null   object        
 12  playerA_DR     8933 non-null   object        
 13  playerA_A%     8933 non-null   object        
 14  playerA_DF%  

In [ ]:
#Elimino columnas no necesarias

df_partidos = df_partidos.drop(columns=['playerA_1stIn','playerB_1stIn'])
df_partidos ["jugadores"] = df_partidos ['Unnamed: 6']
df_partidos = df_partidos.drop(columns=['Unnamed: 6'])

In [ ]:
# Transformar variables en % a tipo numerico
import numpy as np
import pandas as pd

# 1. Transformación Robusta de Porcentajes
variables_pct = ['playerA_A%', 'playerA_DF%', 'playerA_1st%', 'playerA_2nd%',
                 'playerB_A%', 'playerB_DF%', 'playerB_1st%', 'playerB_2nd%']

for col in variables_pct:
    # Verificar primero si la columna existe para evitar errores
    if col in df_partidos.columns:
        # Limpieza: Quitar %, reemplazar guiones y convertir a numérico forzando errores a NaN
        # El .astype(str) asegura que si ya hay números, no falle el .str
        df_partidos[col] = pd.to_numeric(
            df_partidos[col].astype(str).str.replace("%", "").replace('-', np.nan),
            errors='coerce'
        )

# 2. Transformación de Fracciones (BPSvd) - Con manejo de división segura
def procesar_fraccion(df, col_name):
    if col_name not in df.columns:
        return df

    # Extraer numerador y denominador
    split_data = df[col_name].astype(str).str.split("/")

    # Crear series temporales, usando 'coerce' para que "nan" o errores sean NaN
    num = pd.to_numeric(split_data.str[0], errors='coerce')
    den = pd.to_numeric(split_data.str[1], errors='coerce')

    # Calcular porcentaje
    # Si denominador es 0, resultará en np.inf o NaN.
    # En tenis, 0/0 BP suele ser bueno (dominio), pero estadísticamente es indefinido.
    # Lo dejaremos como NaN, pero nos aseguramos de no tener 'inf'.
    stats_pct = num / den

    # Reemplazar infinitos con NaN (por seguridad)
    stats_pct = stats_pct.replace([np.inf, -np.inf], np.nan)

    # Asignar y limpiar
    df[f"{col_name}%"] = stats_pct
    # Opcional: ¿Quieres guardar el denominador? (BP enfrentados es una métrica útil de presión)
    # df[f"{col_name}_faced"] = den

    return df.drop(columns=[col_name])

# Aplicar a ambos jugadores
df_partidos = procesar_fraccion(df_partidos, "playerA_BPSvd")
df_partidos = procesar_fraccion(df_partidos, "playerB_BPSvd")

# 3. Regeneración del Orden Cronológico (Vital)
# Como el dataframe cambió de tamaño al unificar, el "indice" viejo puede estar roto.
# Reordenamos explícitamente y creamos uno nuevo.

if 'Date' in df_partidos.columns and 'Rd' in df_partidos.columns:
    # Definir orden de rondas si no está definido (por seguridad)
    orden_rondas = ["Q1", "Q2", "Q3", "Q4", 'R128', 'R64', 'R32', 'R16', 'QF', "BR", "RR", 'SF', 'F']
    df_partidos["Rd"] = df_partidos["Rd"].astype("category")
    # Solo setear categorias si los valores existen en la columna, para evitar warnings
    df_partidos["Rd"] = df_partidos["Rd"].cat.set_categories(orden_rondas, ordered=True)

    # Ordenar
    df_partidos = df_partidos.sort_values(by=["Date", "Rd"])

    # Crear nuevo índice cronológico limpio
    # (Esto es lo que usarán tus Rolling Windows)
    df_partidos = df_partidos.reset_index(drop=True)
    df_partidos["indice"] = df_partidos.index + 1

    print("Variables transformadas y nuevo índice cronológico generado.")
else:
    print("Variables transformadas. No se pudo reordenar (faltan columnas Date/Rd).")

# Verificación final
print(df_partidos[['playerA_1st%', 'playerA_BPSvd%', 'indice']].head())

Variables transformadas y nuevo índice cronológico generado.
   playerA_1st%  playerA_BPSvd%  indice
0          84.6             NaN       1
1          76.9        1.000000       2
2          69.1        0.625000       3
3          75.9        0.750000       4
4          50.0        0.533333       5


In [ ]:
df_partidos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9003 entries, 0 to 9002
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Date            9003 non-null   datetime64[ns]
 1   Tournament      9003 non-null   object        
 2   Surface         9003 non-null   object        
 3   Rd              9003 non-null   category      
 4   Rk              8987 non-null   float64       
 5   vRk             8942 non-null   float64       
 6   Score           9003 non-null   object        
 7   Time            8766 non-null   object        
 8   indice          9003 non-null   int64         
 9   player_A        9003 non-null   object        
 10  player_B        9003 non-null   object        
 11  playerA_DR      8933 non-null   object        
 12  playerA_A%      8933 non-null   float64       
 13  playerA_DF%     8933 non-null   float64       
 14  playerA_1st%    8933 non-null   float64       
 15  play

In [ ]:
# En  'playerA_BPSvd%', 'playerB_BPSvd%' reemplazo los NAN por 0

df_partidos["playerA_BPSvd%"] = df_partidos["playerA_BPSvd%"].fillna(0)
df_partidos["playerB_BPSvd%"] = df_partidos["playerB_BPSvd%"].fillna(0)

In [ ]:
# Cambio el tipo de las variables

df_partidos['Surface'] = df_partidos['Surface'].astype("category") # Corrected column name Surface

# Handle potential non-numeric values in Rk and vRk by coercing errors
df_partidos['Rk'] = pd.to_numeric(df_partidos['Rk'], errors='coerce').astype("Int64") # Use Int64 to allow for NaNs after coercion
df_partidos['vRk'] = pd.to_numeric(df_partidos['vRk'], errors='coerce').astype("Int64") # Use Int64 to allow for NaNs after coercion

# Display the updated dataframe info to check data types
df_partidos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9003 entries, 0 to 9002
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Date            9003 non-null   datetime64[ns]
 1   Tournament      9003 non-null   object        
 2   Surface         9003 non-null   category      
 3   Rd              9003 non-null   category      
 4   Rk              8987 non-null   Int64         
 5   vRk             8942 non-null   Int64         
 6   Score           9003 non-null   object        
 7   Time            8766 non-null   object        
 8   indice          9003 non-null   int64         
 9   player_A        9003 non-null   object        
 10  player_B        9003 non-null   object        
 11  playerA_DR      8933 non-null   object        
 12  playerA_A%      8933 non-null   float64       
 13  playerA_DF%     8933 non-null   float64       
 14  playerA_1st%    8933 non-null   float64       
 15  play

In [ ]:
#Transformo la variable "score" en "total_juegos" y total_sets

import numpy as np
import pandas as pd
import re

# 1. Filtro de Partidos No Terminados
words_to_filter = ["RET", "DEF", "Def.", "W/O"]
pattern = '|'.join(words_to_filter)
df_partidos = df_partidos[~df_partidos['Score'].astype(str).str.contains(pattern, case=False, na=False)]

# --- LÓGICA ROBUSTA DE EXTRACCIÓN (Corregida para Pandas 2.0+) ---

# 2. Limpieza previa: Eliminar puntos de tie-break "(7)" o "[7]"
scores_limpios = df_partidos['Score'].astype(str).str.replace(r'[\(\[].*?[\)\]]', '', regex=True)

# 3. Extracción de Juegos (Números)
# Esto devuelve un DataFrame con MultiIndex (nivel 0 = índice del partido)
numeros_juegos = scores_limpios.str.extractall(r'(\d+)').astype(int)

# 4. Cálculo de Features (SINTAXIS MODERNA)

# EN LUGAR DE .sum(level=0), USAMOS .groupby(level=0).sum()
total_juegos = numeros_juegos.groupby(level=0).sum()

# EN LUGAR DE .count(level=0), USAMOS .groupby(level=0).count()
# (contamos cuántos números hay para saber los sets)
cantidad_numeros = numeros_juegos.groupby(level=0).count()
total_sets = (cantidad_numeros / 2).astype(int)

# --- ASIGNACIÓN AL DATAFRAME ---

# Como 'total_juegos' es un DataFrame de una columna, seleccionamos la columna 0 para asignar
df_partidos['Total_juegos'] = total_juegos.iloc[:, 0]
df_partidos['Numero_sets'] = total_sets.iloc[:, 0]

# Rellenar NaNs con 0
df_partidos['Total_juegos'] = df_partidos['Total_juegos'].fillna(0).astype(int)
df_partidos['Numero_sets'] = df_partidos['Numero_sets'].fillna(0).astype(int)

# 5. Verificación
print("Ingeniería de Score completada.")
print(df_partidos[['Score', 'Total_juegos', 'Numero_sets']].head(10))

# Validar un partido largo
print("\nVerificación de partidos largos:")
print(df_partidos[df_partidos['Total_juegos'] > 30][['Score', 'Total_juegos']].head())

Ingeniería de Score completada.
                   Score  Total_juegos  Numero_sets
2         3-6 7-6(2) 6-2            30            3
3            6-4 4-6 6-3            29            3
4            6-4 3-6 7-5            31            3
5   7-6(3) 6-7(7) 7-6(5)            39            3
6                7-5 6-4            22            2
7            1-6 6-3 6-1            23            3
8         6-1 6-7(5) 6-3            29            3
9                7-5 6-1            19            2
10        6-7(3) 6-4 6-2            31            3
11               6-4 6-2            18            2

Verificación de partidos largos:
                   Score  Total_juegos
4            6-4 3-6 7-5            31
5   7-6(3) 6-7(7) 7-6(5)            39
10        6-7(3) 6-4 6-2            31
12        2-6 7-6(6) 6-4            31
13        6-7(3) 7-5 6-3            34


In [ ]:
#Creo la variable "duración(min)"
import pandas as pd
import numpy as np

# 1. Asegurar que es string y rellenar nulos iniciales con un valor neutro "0:0"
# Esto evita errores al hacer split en filas vacías
df_partidos["Time"] = df_partidos["Time"].fillna("0:0").astype(str)

# 2. Dividir horas y minutos
# Usamos time_parts para no confundir con el df principal
time_parts = df_partidos["Time"].str.split(":", expand=True)

# 3. Convertir a numérico de forma segura (errors='coerce')
# Si hay basura (ej: "Rain Delay"), lo convertirá a NaN en lugar de romper el código
hours = pd.to_numeric(time_parts[0], errors='coerce').fillna(0)
minutes = pd.to_numeric(time_parts[1], errors='coerce').fillna(0)

# 4. Calcular duración total en minutos
df_partidos["duracion_min"] = (hours * 60) + minutes

# 5. Limpieza final
# Convertir a entero
df_partidos["duracion_min"] = df_partidos["duracion_min"].astype(int)

# Opcional: Volver a poner NaN donde la duración sea 0 (si prefieres tratar los 0 como desconocidos)
# df_partidos["duracion_min"] = df_partidos["duracion_min"].replace(0, np.nan)

# Eliminar la variable original
df_partidos = df_partidos.drop(columns=["Time"], errors='ignore')

# Verificación
print(df_partidos[["duracion_min"]].head())
print(f"Máxima duración encontrada: {df_partidos['duracion_min'].max()} minutos")

   duracion_min
2           122
3           124
4           169
5           180
6            99
Máxima duración encontrada: 700 minutos


In [ ]:
# Si los partidos tienen una duración de 0 min, los descarto

df_partidos = df_partidos[df_partidos["duracion_min"] != 0]

In [ ]:
#Creo una lista con los torneos GrandSlam y ATP 1000

import pandas as pd
import numpy as np

# --- 1. CARGA Y PREPARACIÓN DEL DICCIONARIO MAESTRO ---

# Cargar el CSV de referencia
# Asegúrate de que la ruta sea accesible en tu entorno actual
ruta_csv = "/Users/UJA/Desktop/Datos/lista_torneos.csv"
lista_torneos = pd.read_csv(ruta_csv)

# Limpieza básica del CSV para asegurar coincidencia
# Convertimos a string y quitamos espacios extra
lista_torneos['tourney_name'] = lista_torneos['tourney_name'].astype(str).str.strip()
lista_torneos['tourney_level'] = lista_torneos['tourney_level'].astype(str).str.strip()

# CREAMOS UN DICCIONARIO (Hash Map)
# Esto es mucho más rápido que iterar listas.
# Clave: Nombre del Torneo -> Valor: Nivel ('G', 'M', 'A', 'C', 'F')
mapa_niveles = dict(zip(lista_torneos['tourney_name'], lista_torneos['tourney_level']))

print(f"Se cargaron {len(mapa_niveles)} torneos del archivo CSV.")

# --- 2. APLICACIÓN AL DATAFRAME DE PARTIDOS ---

# A) Mapeo Directo
# Buscamos el nombre del torneo de cada partido en nuestro diccionario
df_partidos['Tier_Code'] = df_partidos['Tournament'].map(mapa_niveles)

# --- 3. SISTEMA DE SEGURIDAD (FALLBACK) ---
# Es posible que 'df_partidos' tenga torneos que NO están en tu 'lista_torneos.csv'.
# Esos aparecerán como NaN. Debemos rellenarlos usando lógica de texto.

def rellenar_niveles_faltantes(row):
    # Si ya tiene código (vino del CSV), lo mantenemos
    if pd.notna(row['Tier_Code']):
        return row['Tier_Code']

    # Si no estaba en el CSV, adivinamos por el nombre
    name = str(row['Tournament']).lower()

    if 'finals' in name: return 'F'
    if 'ch' in name or 'challenger' in name: return 'C'

    # Por defecto, si no sabemos qué es, asumimos que es un ATP regular ('A')
    # (Es más seguro que asumir que es un Grand Slam)
    return 'A'

# Rellenamos los huecos
na_count = df_partidos['Tier_Code'].isna().sum()
if na_count > 0:
    print(f"¡Atención! {na_count} partidos no se encontraron en el CSV. Se imputarán por nombre.")
    df_partidos['Tier_Code'] = df_partidos.apply(rellenar_niveles_faltantes, axis=1)

# --- 4. CREACIÓN DE VARIABLES NUMÉRICAS PARA EL MODELO ---

# Definimos la Jerarquía de Importancia (Vital para Value Betting)
tier_mapping = {
    'C': 1,  # Challenger / Futures
    'A': 2,  # ATP 250 / 500
    'M': 3,  # Masters 1000
    'F': 4,  # Tour Finals
    'G': 5   # Grand Slam
}

# Mapear a números (Esta es la variable que usará XGBoost)
df_partidos['Tier_Numeric'] = df_partidos['Tier_Code'].map(tier_mapping).fillna(2).astype(int)

# Crear Best_of_5 (Solo GS son a 5 sets)
df_partidos['Best_of_5'] = (df_partidos['Tier_Code'] == 'G').astype(int)

# --- 5. VERIFICACIÓN ---
print("\nDistribución final de niveles (Tier_Code):")
print(df_partidos['Tier_Code'].value_counts())

# Eliminar las variables antiguas si existían para no duplicar
cols_drop = ["tourney_level_G", "tourney_level_M"]
df_partidos = df_partidos.drop(columns=cols_drop, errors='ignore')

print("\nEjemplo de datos procesados:")
print(df_partidos[['Tournament', 'Tier_Code', 'Tier_Numeric', 'Best_of_5']].sample(5))

Se cargaron 1961 torneos del archivo CSV.
¡Atención! 5717 partidos no se encontraron en el CSV. Se imputarán por nombre.

Distribución final de niveles (Tier_Code):
Tier_Code
C    5717
A    1454
M     701
G     514
Name: count, dtype: int64

Ejemplo de datos procesados:
                      Tournament Tier_Code  Tier_Numeric  Best_of_5
4604  Santa Cruz de la Sierra CH         C             1          0
3517                   Bogota CH         C             1          0
6533                     US Open         G             5          1
342              Australian Open         G             5          1
3411                     Wuxi CH         C             1          0


In [ ]:
# Creación de playerA_won
import numpy as np
import numpy as np
import pandas as pd
import re

print("Iniciando creación de target (Método .apply() robusto)...")

# --- 1. EXTRACCIÓN (Tu lógica original) ---
extracted_data = df_partidos['jugadores'].str.extract(r'(.*)(d\.|l\.)(.*)')
extracted_data.columns = ['Player_Info', 'Result', 'Opponent_Info']

# --- 2. LIMPIEZA (Tu lógica original) ---
CLEAN_REGEX = r'^\(.*?\)|\s*\[.*?\]$'

df_partidos['Parsed_Left'] = extracted_data['Player_Info'].str.replace(CLEAN_REGEX, '', regex=True).str.strip()
df_partidos['Parsed_Right'] = extracted_data['Opponent_Info'].str.replace(CLEAN_REGEX, '', regex=True).str.strip()
df_partidos['Result'] = extracted_data['Result']

# Limpiamos los nombres completos de A y B para una comparación justa
df_partidos['player_A_cleaned'] = df_partidos['player_A'].astype(str).str.replace(CLEAN_REGEX, '', regex=True).str.strip()
df_partidos['player_B_cleaned'] = df_partidos['player_B'].astype(str).str.replace(CLEAN_REGEX, '', regex=True).str.strip()


# --- 3. CREACIÓN DE TARGET (Función .apply()) ---

def encontrar_target(row):
    # Nombres completos
    player_A = row['player_A_cleaned']

    # Nombres parciales (del string 'jugadores')
    parsed_L = row['Parsed_Left']
    parsed_R = row['Parsed_Right']

    # Resultado (d. o l.)
    result = row['Result']

    # Seguridad: Si falta algún dato, no podemos saber
    if pd.isna(player_A) or pd.isna(parsed_L) or pd.isna(parsed_R) or pd.isna(result):
        return np.nan

    try:
        # Caso 1: player_A es el jugador de la izquierda
        # (ej. "Monfils" in "Gael Monfils")
        if parsed_L in player_A:
            if result == 'd.':
                return 1 # A (izquierda) ganó
            elif result == 'l.':
                return 0 # A (izquierda) perdió

        # Caso 2: player_A es el jugador de la derecha
        elif parsed_R in player_A:
            if result == 'd.':
                return 0 # A (derecha) perdió (porque el de la izquierda ganó)
            elif result == 'l.':
                return 1 # A (derecha) ganó (porque el de la izquierda perdió)

        # Caso 3: No hay match. (Colisión de nombres o error de datos)
        return np.nan # Es más seguro que asumir un 0

    except TypeError:
        # Por si acaso 'in' falla (ej. parsed_L es un float 0.0)
        return np.nan

# Aplicamos la función fila por fila
df_partidos['playerA_won'] = df_partidos.apply(encontrar_target, axis=1)

# --- 4. VERIFICACIÓN Y LIMPIEZA ---
print("\n--- ¡Éxito! 'playerA_won' creado correctamente. ---")
print("Distribución del Target (Excluyendo NaNs de 'match'):")
print(df_partidos['playerA_won'].value_counts(normalize=True, dropna=True))

# Opcional: ¿Cuántos partidos perdimos por no encontrar match?
print("\nPartidos descartados (NaNs):")
print(df_partidos['playerA_won'].isna().sum())

# Limpieza final
cols_a_borrar = [
    'jugadores', 'Result', 'Parsed_Left', 'Parsed_Right',
    'player_A_cleaned', 'player_B_cleaned',
    'Player_Name', 'Opponent_Name', 'player_A_apellido' # De tu código original
]
df_partidos = df_partidos.drop(columns=cols_a_borrar, errors='ignore')

print("\nVerificación de datos finales:")
display(df_partidos[['player_A', 'player_B', 'playerA_won']].head())

Iniciando creación de target (Método .apply() robusto)...

--- ¡Éxito! 'playerA_won' creado correctamente. ---
Distribución del Target (Excluyendo NaNs de 'match'):
playerA_won
1    0.66301
0    0.33699
Name: proportion, dtype: float64

Partidos descartados (NaNs):
0

Verificación de datos finales:


,player_A,player_B,playerA_won
2,Hugo Gaston,Yannick Hanfmann,0
3,Adam Walton,Dusan Lajovic,1
4,Damir Dzumhur,James Mccabe,0
5,James Duckworth,Pavel Kotov,1
6,Taro Daniel,Manuel Guinard,0


In [ ]:
import pandas as pd
import numpy as np

print("Iniciando Paso 1: Limpieza Numérica Robusta...")

# Lista de todas las columnas que deben ser números
columnas_a_limpiar = [
    'playerA_DR', 'playerA_A%', 'playerA_DF%', 'playerA_1st%', 'playerA_2nd%',
    'playerB_DR', 'playerB_A%', 'playerB_DF%', 'playerB_1st%', 'playerB_2nd%',
    'playerA_1stIn', 'playerB_1stIn' # Añade estas si existen
]

for col in columnas_a_limpiar:
    if col in df_partidos.columns:
        # 1. Reemplazar guiones o texto común
        df_partidos[col] = df_partidos[col].astype(str).str.replace("%", "").replace('-', np.nan)

        # 2. Convertir a numérico (LA PARTE CLAVE)
        # errors='coerce' convertirá la cadena '0.722.431...' y cualquier otra
        # cosa que no sea un número en NaN (Not a Number).
        df_partidos[col] = pd.to_numeric(df_partidos[col], errors='coerce')

print("Limpieza numérica completada. Los datos corruptos ahora son NaN.")

# Verificar los tipos de datos (Dtypes). Deberían ser 'float64'
print("\n--- Nuevos Tipos de Datos (Dtypes) ---")
cols_existentes = [c for c in columnas_a_limpiar if c in df_partidos.columns]
if cols_existentes:
    print(df_partidos[cols_existentes].info())
else:
    print("Asegúrate de que las columnas de estadísticas (playerA_DR, etc.) existen.")

Iniciando Paso 1: Limpieza Numérica Robusta...
Limpieza numérica completada. Los datos corruptos ahora son NaN.

--- Nuevos Tipos de Datos (Dtypes) ---
<class 'pandas.core.frame.DataFrame'>
Index: 8386 entries, 2 to 9000
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   playerA_DR    8386 non-null   float64
 1   playerA_A%    8386 non-null   float64
 2   playerA_DF%   8386 non-null   float64
 3   playerA_1st%  8386 non-null   float64
 4   playerA_2nd%  8385 non-null   float64
 5   playerB_DR    8386 non-null   float64
 6   playerB_A%    8386 non-null   float64
 7   playerB_DF%   8386 non-null   float64
 8   playerB_1st%  8386 non-null   float64
 9   playerB_2nd%  8385 non-null   float64
dtypes: float64(10)
memory usage: 720.7 KB
None


In [ ]:
#Codificando variables categóricas de forma óptima para XGBoost.
from sklearn.preprocessing import LabelEncoder
import pandas as pd

print("Codificando variables (con corrección para TypeError)...")

# --- 1. Mapeo Ordinal para 'Rd' (Ronda) ---
# (Esto ya debería estar bien, pero lo mantenemos por coherencia)
if 'Rd' in df_partidos.columns:
    round_mapping = {
        'Q1': 1, 'Q2': 2, 'Q3': 3, 'Q4': 4,
        'R128': 5, 'R64': 6, 'R32': 7, 'R16': 8,
        'BR': 9, 'RR': 10, 'QF': 11, 'SF': 12, 'F': 13
    }
    df_partidos['Rd_Ordinal'] = df_partidos['Rd'].map(round_mapping)

# --- 2. Codificación de Etiquetas para 'Surface' (Superficie) ---

if 'Surface' in df_partidos.columns:
    # --- ¡LA CORRECCIÓN ESTÁ AQUÍ! ---
    # 1. Forzamos la columna a tipo 'str' (object).
    #    Esto "rompe" el tipo 'category' y nos permite manipularla.
    df_partidos['Surface'] = df_partidos['Surface'].astype(str)

    # 2. Reemplazamos los 'nan' (que ahora son strings) por 'Unknown'
    #    (Usar .replace() es más seguro que .fillna() después de .astype(str))
    df_partidos['Surface'] = df_partidos['Surface'].replace('nan', 'Unknown')

    # 3. Ahora LabelEncoder funciona perfectamente
    surface_encoder = LabelEncoder()
    df_partidos['Surface_Encoded'] = surface_encoder.fit_transform(df_partidos['Surface'])

    print("\nMapeo de 'Surface' creado:")
    print(dict(zip(surface_encoder.classes_, surface_encoder.transform(surface_encoder.classes_))))

# --- 3. Limpieza ---
# Eliminamos las columnas de string originales
df_partidos = df_partidos.drop(columns=['Surface', 'Rd'], errors='ignore')

print("\nCodificación completada.")
print(df_partidos[['Rd_Ordinal', 'Surface_Encoded']].head())

Codificando variables (con corrección para TypeError)...

Mapeo de 'Surface' creado:
{'Clay': np.int64(0), 'Grass': np.int64(1), 'Hard': np.int64(2)}

Codificación completada.
  Rd_Ordinal  Surface_Encoded
2          1                2
3          1                2
4          1                2
5          1                2
6          1                2


In [ ]:
import pandas as pd
import numpy as np

# (Asumimos que 'df_partidos_2000_2024_adaptado' ya existe)
print("Iniciando la extracción de datos biográficos (2000-2024)...")

# --- 1. Extraer datos de Player A ---
df_A = df_partidos_2000_2024_adaptado[['Date', 'player_A', 'ht_A', 'hand_A_encoded', 'age_A']].copy()
df_A.columns = ['match_date', 'player_name', 'height', 'hand', 'age_at_match']

# --- 2. Extraer datos de Player B ---
df_B = df_partidos_2000_2024_adaptado[['Date', 'player_B', 'ht_B', 'hand_B_encoded', 'age_B']].copy()
df_B.columns = ['match_date', 'player_name', 'height', 'hand', 'age_at_match']

# --- 3. Combinar A y B en una sola lista ---
df_bio_combined = pd.concat([df_A, df_B], ignore_index=True)
print(f"Filas totales (duplicadas) encontradas: {len(df_bio_combined)}")

# --- 4. Limpiar y De-duplicar ---
# Ordenamos por fecha (más reciente primero)
# Esto asegura que nos quedamos con la altura/mano más RECIENTE de un jugador
df_bio_combined = df_bio_combined.sort_values(by='match_date', ascending=False)

# Limpiamos NaNs de las claves
df_bio_combined = df_bio_combined.dropna(subset=['player_name', 'age_at_match', 'height', 'hand'])
# Eliminamos filas donde la mano es 'U' (Unknown) si es posible
df_bio_combined = df_bio_combined[df_bio_combined['hand'] != 'U']

# Creamos la clave de jugador
df_bio_combined['player_key'] = df_bio_combined['player_name'].astype(str).str.strip().str.lower()
df_bio_combined = df_bio_combined.drop_duplicates(subset=['player_key'], keep='first')

print(f"Jugadores únicos encontrados: {len(df_bio_combined)}")

# --- 5. Ingeniería Inversa de Fecha de Nacimiento (Birthdate) ---
# Birthdate = Fecha_Partido - (Edad * 365.25)
df_bio_combined['Birthdate'] = df_bio_combined['match_date'] - pd.to_timedelta(df_bio_combined['age_at_match'] * 365.25, unit='D')
# Redondeamos al día (es una aproximación, pero muy cercana)
df_bio_combined['Birthdate'] = pd.to_datetime(df_bio_combined['Birthdate'].dt.date)

# --- 6. Formatear para coincidir con tu 'df_bio' (del Turno 170) ---
# (Columnas: 'Player', 'Birthdate', 'hand', 'altura')
df_bio_extraido = df_bio_combined[['player_name', 'Birthdate', 'hand', 'height']].copy()
df_bio_extraido = df_bio_extraido.rename(columns={
    'player_name': 'Player',
    'height': 'altura'
})

print("\n--- ¡Éxito! Nuevo DataFrame de Biografías Creado ---")
display(df_bio_extraido.head())

Iniciando la extracción de datos biográficos (2000-2024)...
Filas totales (duplicadas) encontradas: 497790
Jugadores únicos encontrados: 2734

--- ¡Éxito! Nuevo DataFrame de Biografías Creado ---


,Player,Birthdate,hand,altura
497789,Alex Michelsen,2004-08-30,R,193.0
248880,Joao Fonseca,2006-08-30,R,185.0
248883,Jakub Mensik,2005-10-06,R,193.0
248884,Learner Tien,2005-12-18,L,180.0
248885,Arthur Fils,2004-06-18,R,185.0


In [ ]:
#Creación de "playerA_edad" y player_B_edad

import pandas as pd
import numpy as np


df_bio = df_bio_extraido.copy()

   # Renombrar por si acaso
df_bio = df_bio.rename(columns={

        'Player': 'player_name',

        'Birthdate': 'date_of_birth',

        'altura': 'height',

        'hand': 'hand'

    })

# # --- 2. Limpieza de Claves para el Merge ---

# # Limpiamos la clave del archivo BIO (minúsculas, sin espacios)
# # (Esto previene los errores de merge que tuvimos antes)
# df_bio['player_key'] = df_bio['player_name'].astype(str).str.strip().str.lower()
# df_bio = df_bio.drop_duplicates(subset=['player_key'], keep='first')

# # Asegurarnos de que las claves de df_partidos están igual de limpias
# # (Aunque ya lo hicimos en el Turno 62, lo repetimos por seguridad)
# df_partidos['player_A'] = df_partidos['player_A'].astype(str).str.strip().str.lower()
# df_partidos['player_B'] = df_partidos['player_B'].astype(str).str.strip().str.lower()


# --- 3. Unir (Merge) la Información Biográfica ---

print("Uniendo biografías a Player A y Player B...")

# Columnas BIO que queremos añadir
bio_cols = ['player_name', 'date_of_birth', 'height', 'hand']

# Merge para Player A
df_partidos = pd.merge(
    df_partidos,
    df_bio[bio_cols],
    left_on='player_A',
    right_on='player_name',
    how='left'
)
# Renombrar las nuevas columnas (ej. 'height_A')
df_partidos = df_partidos.rename(columns={
    'date_of_birth': 'dob_A',
    'height': 'ht_A',
    'hand': 'hand_A'
}).drop(columns='player_name')

# Merge para Player B
df_partidos = pd.merge(
    df_partidos,
    df_bio[bio_cols],
    left_on='player_B',
    right_on='player_name',
    how='left'
)
# Renombrar las nuevas columnas (ej. 'height_B')
df_partidos = df_partidos.rename(columns={
    'date_of_birth': 'dob_B',
    'height': 'ht_B',
    'hand': 'hand_B'
}).drop(columns='player_name')


# --- 4. Cálculo de Edad Dinámica ---

print("Calculando edad dinámica (en el día del partido)...")

# Asegurarnos de que la fecha del partido es datetime
df_partidos['Date'] = pd.to_datetime(df_partidos['Date'])

# Calcular edad como un float (ej. 32.4 años)
# (Fecha Partido - Fecha Nacimiento) / 365.25 días
df_partidos['age_A'] = (df_partidos['Date'] - df_partidos['dob_A']).dt.days / 365.25
df_partidos['age_B'] = (df_partidos['Date'] - df_partidos['dob_B']).dt.days / 365.25


# --- 5. Codificación de 'hand' (Mano) ---
# Los modelos necesitan números. Convertimos 'L' y 'R' a 1 y 0.
# (R=Right, L=Left, U=Unknown/Ambidiestro)

def encode_hand(hand_series):
    return hand_series.str.upper().map({
        'R': 1,  # Diestro
        'L': -1, # Zurdo
        'U': 0,  # Desconocido/Ambi
    }).fillna(0) # Rellenar NaNs con 0 (Desconocido)

df_partidos['hand_A_encoded'] = encode_hand(df_partidos['hand_A'])
df_partidos['hand_B_encoded'] = encode_hand(df_partidos['hand_B'])


# --- 6. (Opcional) Creación de Features Diferenciales ---
# Estas son features muy potentes para el modelo
print("Creando features diferenciales (ht_diff, age_diff)...")
df_partidos['ht_diff'] = df_partidos['ht_A'] - df_partidos['ht_B']
df_partidos['age_diff'] = df_partidos['age_A'] - df_partidos['age_B']
# Ventaja de mano (ej. +2 si A es Zurdo y B es Diestro)
df_partidos['hand_diff'] = df_partidos['hand_A_encoded'] - df_partidos['hand_B_encoded']
df_partidos['ht_sum'] = df_partidos['ht_A'] + df_partidos['ht_B']

# --- 7. Limpieza Final ---
cols_bio_originales = ['dob_A', 'dob_B', 'hand_A', 'hand_B']
df_partidos = df_partidos.drop(columns=cols_bio_originales, errors='ignore')

print("\n--- ¡Éxito! Biografías y Edades añadidas ---")
cols_a_mostrar = [
    'player_A', 'ht_A', 'age_A', 'hand_A_encoded',
    'player_B', 'ht_B', 'age_B', 'hand_B_encoded',
    'ht_diff', 'age_diff', 'hand_diff'
]
display(df_partidos[cols_a_mostrar].head())

Uniendo biografías a Player A y Player B...
Calculando edad dinámica (en el día del partido)...
Creando features diferenciales (ht_diff, age_diff)...

--- ¡Éxito! Biografías y Edades añadidas ---


,player_A,ht_A,age_A,hand_A_encoded,player_B,ht_B,age_B,hand_B_encoded,ht_diff,age_diff,hand_diff
0,Hugo Gaston,173.0,24.273785,-1.0,Yannick Hanfmann,193.0,33.073238,1.0,-20.0,-8.799452,-2.0
1,Adam Walton,183.0,25.653662,1.0,Dusan Lajovic,183.0,34.475017,1.0,0.0,-8.821355,0.0
2,Damir Dzumhur,175.0,32.616016,1.0,James Mccabe,188.0,21.415469,1.0,-13.0,11.200548,0.0
3,James Duckworth,183.0,32.955510,1.0,Pavel Kotov,191.0,26.072553,1.0,-8.0,6.882957,0.0
4,Taro Daniel,191.0,31.854894,1.0,Manuel Guinard,198.0,29.073238,1.0,-7.0,2.781656,0.0


In [ ]:
df_partidos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8386 entries, 0 to 8385
Data columns (total 39 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Date             8386 non-null   datetime64[ns]
 1   Tournament       8386 non-null   object        
 2   Rk               8371 non-null   Int64         
 3   vRk              8325 non-null   Int64         
 4   Score            8386 non-null   object        
 5   indice           8386 non-null   int64         
 6   player_A         8386 non-null   object        
 7   player_B         8386 non-null   object        
 8   playerA_DR       8386 non-null   float64       
 9   playerA_A%       8386 non-null   float64       
 10  playerA_DF%      8386 non-null   float64       
 11  playerA_1st%     8386 non-null   float64       
 12  playerA_2nd%     8385 non-null   float64       
 13  playerB_DR       8386 non-null   float64       
 14  playerB_A%       8386 non-null   float64

In [ ]:
#Combinación dataframes

#"Actualizar" el DataFrame 2000-2024

import pandas as pd
import numpy as np

# 1. Definir la función de codificación
# (Esta función faltaba en el script de 'Adaptación' del Turno 161)
def encode_hand(hand_series):
    # Asegurarse de que es string, por si acaso
    return hand_series.astype(str).str.upper().map({
        'R': 1,  # Diestro
        'L': -1, # Zurdo
        'U': 0   # Desconocido/Ambi
    }).fillna(0) # Rellenar NaNs (y 'None') con 0

# --- 2. Arreglar df_partidos_2000_2024_adaptado ---
print("Arreglando el dataframe 2000-2024...")
df_partidos_2000_2024_adaptado['hand_A_encoded'] = encode_hand(df_partidos_2000_2024_adaptado['hand_A_encoded'])
df_partidos_2000_2024_adaptado['hand_B_encoded'] = encode_hand(df_partidos_2000_2024_adaptado['hand_B_encoded'])

# Ahora SÍ podemos calcular los _diff (Paso 1 de Turno 172)
print("Calculando columnas '_diff' para 2000-2024...")
df_partidos_2000_2024_adaptado['ht_diff'] = df_partidos_2000_2024_adaptado['ht_A'] - df_partidos_2000_2024_adaptado['ht_B']
df_partidos_2000_2024_adaptado['age_diff'] = df_partidos_2000_2024_adaptado['age_A'] - df_partidos_2000_2024_adaptado['age_B']
df_partidos_2000_2024_adaptado['hand_diff'] = df_partidos_2000_2024_adaptado['hand_A_encoded'] - df_partidos_2000_2024_adaptado['hand_B_encoded']
df_partidos_2000_2024_adaptado['ht_sum'] = df_partidos_2000_2024_adaptado['ht_A'] + df_partidos_2000_2024_adaptado['ht_B']

# --- 3. Arreglar df_partidos (2025) ---
print("Arreglando el dataframe 2025...")
df_partidos['hand_A_encoded'] = encode_hand(df_partidos['hand_A_encoded'])
df_partidos['hand_B_encoded'] = encode_hand(df_partidos['hand_B_encoded'])

# Re-calcular 'hand_diff' para 2025 para estar 100% seguros
# (Las otras _diff (ht, age) ya estaban bien)
df_partidos['hand_diff'] = df_partidos['hand_A_encoded'] - df_partidos['hand_B_encoded']
print("Columnas 'hand' arregladas y 'hand_diff' (re)calculada para 2025.")

# Cambio el tipo de duracion_min en df_partidos

df_partidos["duracion_min"]= df_partidos["duracion_min"].astype(float, errors= 'raise')

# --- 4. Armonizar y Combinar (Paso 2 de Turno 172) ---
print("\nPaso 2: Armonizando columnas para la combinación...")
cols_2000_2024 = set(df_partidos_2000_2024_adaptado.columns)
cols_2025 = set(df_partidos.columns)

columnas_comunes = list(cols_2000_2024.intersection(cols_2025))
print(f"Se usarán {len(columnas_comunes)} columnas comunes.")
# (Esto excluirá 'Round' (solo en 2000-2024) y 'Tier_Code' (solo en 2025))

df_1 = df_partidos_2000_2024_adaptado[columnas_comunes]
df_2 = df_partidos[columnas_comunes]

print("Combinando los dataframes (2000-2024) y (2025)...")
df_partidos_maestro = pd.concat([df_1, df_2], ignore_index=True)

# 5. Ordenar por fecha (¡CRUCIAL!) y re-crear el índice
df_partidos_maestro = df_partidos_maestro.sort_values(by='Date').reset_index(drop=True)
df_partidos_maestro['indice'] = df_partidos_maestro.index

print(f"\n--- ¡DataFrame Maestro (2000-2025) Creado! ---")
print(f"Total de filas: {len(df_partidos_maestro)}")
print(f"Fecha de inicio: {df_partidos_maestro['Date'].min()}")
print(f"Fecha de fin: {df_partidos_maestro['Date'].max()}")
print(f"Columnas finales ({len(df_partidos_maestro.columns)}): {df_partidos_maestro.columns.to_list()}")

df_partidos = df_partidos_maestro.copy()

Arreglando el dataframe 2000-2024...
Calculando columnas '_diff' para 2000-2024...
Arreglando el dataframe 2025...
Columnas 'hand' arregladas y 'hand_diff' (re)calculada para 2025.

Paso 2: Armonizando columnas para la combinación...
Se usarán 38 columnas comunes.
Combinando los dataframes (2000-2024) y (2025)...

--- ¡DataFrame Maestro (2000-2025) Creado! ---
Total de filas: 257281
Fecha de inicio: 2000-01-03 00:00:00
Fecha de fin: 2025-11-10 00:00:00
Columnas finales (38): ['age_B', 'Rk', 'Date', 'Score', 'playerA_won', 'playerA_2nd%', 'playerB_A%', 'ht_A', 'vRk', 'Rd_Ordinal', 'Tier_Numeric', 'playerA_BPSvd%', 'age_diff', 'player_B', 'Tournament', 'hand_diff', 'playerA_DF%', 'Total_juegos', 'ht_B', 'age_A', 'playerB_BPSvd%', 'ht_diff', 'Best_of_5', 'playerA_1st%', 'indice', 'Surface_Encoded', 'playerB_1st%', 'playerB_DR', 'playerB_DF%', 'playerB_2nd%', 'ht_sum', 'playerA_A%', 'hand_B_encoded', 'Numero_sets', 'hand_A_encoded', 'playerA_DR', 'duracion_min', 'player_A']


In [ ]:
df_partidos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 257281 entries, 0 to 257280
Data columns (total 38 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   age_B            256359 non-null  float64       
 1   Rk               252437 non-null  Float64       
 2   Date             257281 non-null  datetime64[ns]
 3   Score            257263 non-null  object        
 4   playerA_won      257281 non-null  int64         
 5   playerA_2nd%     257280 non-null  float64       
 6   playerB_A%       257281 non-null  float64       
 7   ht_A             236551 non-null  float64       
 8   vRk              252630 non-null  Float64       
 9   Rd_Ordinal       257281 non-null  float64       
 10  Tier_Numeric     257281 non-null  int64         
 11  playerA_BPSvd%   257281 non-null  float64       
 12  age_diff         255964 non-null  float64       
 13  player_B         257281 non-null  object        
 14  Tournament       257

In [ ]:
# Promedios ponderados para numero sets y total juegos

import pandas as pd
import numpy as np

print("Calculando 'Forma Reciente' (Fatiga) con SEGREGACIÓN (Corregido con Merge)...")

# --- 1. Crear las columnas de fatiga segregadas ---
# (Esta parte estaba bien)
df_partidos['Juegos_Bo3'] = np.where(df_partidos['Best_of_5'] == 0, df_partidos['Total_juegos'], 0)
df_partidos['Juegos_Bo5'] = np.where(df_partidos['Best_of_5'] == 1, df_partidos['Total_juegos'], 0)
df_partidos['Sets_Bo3'] = np.where(df_partidos['Best_of_5'] == 0, df_partidos['Numero_sets'], 0)
df_partidos['Sets_Bo5'] = np.where(df_partidos['Best_of_5'] == 1, df_partidos['Numero_sets'], 0)

# --- 2. Reorganizar (Unstack) ---
# (Esta parte estaba bien)
df_temp = pd.DataFrame({
    'indice': pd.concat([df_partidos['indice'], df_partidos['indice']]),
    'Player': pd.concat([df_partidos['player_A'], df_partidos['player_B']]),
    'Juegos_Bo3': pd.concat([df_partidos['Juegos_Bo3'], df_partidos['Juegos_Bo3']]),
    'Juegos_Bo5': pd.concat([df_partidos['Juegos_Bo5'], df_partidos['Juegos_Bo5']]),
    'Sets_Bo3': pd.concat([df_partidos['Sets_Bo3'], df_partidos['Sets_Bo3']]),
    'Sets_Bo5': pd.concat([df_partidos['Sets_Bo5'], df_partidos['Sets_Bo5']])
})

# Ordenar por Jugador y Orden Cronológico REAL (índice)
df_temp = df_temp.sort_values(by=['Player', 'indice'])

# --- 3. Calcular EWM (Media Móvil Exponencial) ---
# (Esta parte, de la corrección del Turno 51, estaba bien)
span = 10
columnas_fatiga = ['Juegos_Bo3', 'Juegos_Bo5', 'Sets_Bo3', 'Sets_Bo5']

for col in columnas_fatiga:
    ewm_series = df_temp.groupby('Player')[col].ewm(
        span=span, min_periods=1, adjust=False
    ).mean().reset_index(level=0, drop=True)

    df_temp[f'form_actual_{col}'] = ewm_series
    df_temp[f'form_prev_{col}'] = df_temp.groupby('Player')[f'form_actual_{col}'].shift(1)

# --- 4. Manejar "Cold Starts" (Primer partido) ---
# (Esta parte estaba bien)
df_temp.fillna(0, inplace=True)

# --- 5. Unir los resultados al DataFrame original (MÉTODO MERGE) ---
print("Uniendo (merge) resultados de Player A...")

# Seleccionar solo las columnas de 'forma' que necesitamos de df_temp
cols_forma = ['indice', 'Player'] + [f'form_prev_{col}' for col in columnas_fatiga]
df_temp_forma = df_temp[cols_forma]

# Unir para Player A
df_partidos = pd.merge(
    df_partidos,
    df_temp_forma,
    left_on=['indice', 'player_A'],
    right_on=['indice', 'Player'],
    how='left'
)

# Renombrar las columnas unidas para Player A
rename_map_A = {f'form_prev_{col}': f'avg_{col.lower()}_A' for col in columnas_fatiga}
df_partidos = df_partidos.rename(columns=rename_map_A)
df_partidos = df_partidos.drop(columns='Player') # Borrar la columna 'Player' de df_temp

print("Uniendo (merge) resultados de Player B...")
# Unir para Player B
df_partidos = pd.merge(
    df_partidos,
    df_temp_forma,
    left_on=['indice', 'player_B'],
    right_on=['indice', 'Player'],
    how='left'
)

# Renombrar las columnas unidas para Player B
rename_map_B = {f'form_prev_{col}': f'avg_{col.lower()}_B' for col in columnas_fatiga}
df_partidos = df_partidos.rename(columns=rename_map_B)
df_partidos = df_partidos.drop(columns='Player') # Borrar la columna 'Player' de df_temp

# Limpieza de columnas temporales en df_partidos
cols_a_borrar_orig = ['Juegos_Bo3', 'Juegos_Bo5', 'Sets_Bo3', 'Sets_Bo5']
df_partidos = df_partidos.drop(columns=cols_a_borrar_orig, errors='ignore')

print("Cálculo de EWM segregado y unión (merge) completado.")
display(df_partidos[['player_A', 'player_B', 'avg_juegos_bo3_A', 'avg_juegos_bo5_A']].head())

Calculando 'Forma Reciente' (Fatiga) con SEGREGACIÓN (Corregido con Merge)...
Uniendo (merge) resultados de Player A...
Uniendo (merge) resultados de Player B...
Cálculo de EWM segregado y unión (merge) completado.


,player_A,player_B,avg_juegos_bo3_A,avg_juegos_bo5_A
0,Ivo Heuberger,Cristiano Caratti,0.0,0.0
1,Younes El Aynaoui,Antony Dupuis,0.0,0.0
2,Fernando Vicente,Daniel Vacek,0.0,0.0
3,Fabrice Santoro,Rainer Schuettler,0.0,0.0
4,Younes El Aynaoui,Rainer Schuettler,0.0,0.0


In [ ]:
#Creación nuevas variables numéricas

import pandas as pd
import numpy as np

print("Creando features de Ranking (Forma Robusta)...")

# Asumimos que las columnas de ranking se llaman 'Rk' (A) y 'vRk' (B)
# Si tu código de unificación (Turno 21) creó 'playerA_Rk', ajusta los nombres

if 'Rk' in df_partidos.columns and 'vRk' in df_partidos.columns:

    # 1. Convertir a numérico, forzando errores (texto, 'NR', etc.) a NaN
    rk_A = pd.to_numeric(df_partidos['Rk'], errors='coerce')
    rk_B = pd.to_numeric(df_partidos['vRk'], errors='coerce')

    # 2. Rellenar NaNs (sin ranking) con un valor alto (ej. 1000)
    #    Esto es mejor que 0, ya que sin ranking es "peor" que ser #1
    rk_A = rk_A.fillna(1000)
    rk_B = rk_B.fillna(1000)

    # 3. Calcular Diferencia
    # (Ej. 10 - 100 = -90. Un valor negativo es bueno para A)
    df_partidos["rank_diff"] = rk_A - rk_B

    # 4. Calcular Ratio (evitando división por cero)
    # (Ej. 10 / 100 = 0.1. Un valor bajo es bueno para A)
    df_partidos["rank_ratio"] = rk_A / (rk_B + 1) # +1 por si rk_B es 0

    print("Features 'rank_diff' y 'rank_ratio' creadas.")

else:
    print("Error: No se encontraron las columnas 'Rk' y 'vRk'.")
    print("Por favor, renombra las columnas de ranking de tus jugadores A y B.")


Creando features de Ranking (Forma Robusta)...
Features 'rank_diff' y 'rank_ratio' creadas.


In [ ]:
#Creación variables H2H % victoria

df_partidos = df_partidos.sort_values(by='indice')

# 3. Inicializar el registro histórico
from collections import defaultdict
# historial_enfrentamientos: { jugador1_id: { jugador2_id: num_partidos } }
historial_enfrentamientos = defaultdict(lambda: defaultdict(int))
# historial_victorias: { jugador1_id: { jugador2_id: num_victorias_jugador1_vs_jugador2 } }
historial_victorias = defaultdict(lambda: defaultdict(int))

# Listas para almacenar las nuevas características
player_A_vs_B_matches_list = []
player_A_wins_vs_B_list = []
player_B_wins_vs_A_list = []

print("Procesando partidos cronológicamente para generar características históricas...")

# 4. Iterar por cada fila (partido) del DataFrame ordenado
for index, row in df_partidos.iterrows():
    player_A = row['player_A']
    player_B = row['player_B']
    player_A_won = row['playerA_won']

    # --- Calcular características históricas ANTES de actualizar el registro ---

    # Conteo de enfrentamientos históricos entre A y B (simétrico, no importa el orden de A y B en filas anteriores)
    current_matches = max(
        historial_enfrentamientos[player_A][player_B],
        historial_enfrentamientos[player_B][player_A]
    )
    player_A_vs_B_matches_list.append(current_matches)

    # Victorias históricas de playerA contra playerB
    current_A_wins_vs_B = historial_victorias[player_A][player_B]
    player_A_wins_vs_B_list.append(current_A_wins_vs_B)

    # Victorias históricas de playerB contra playerA
    current_B_wins_vs_A = historial_victorias[player_B][player_A]
    player_B_wins_vs_A_list.append(current_B_wins_vs_A)

    # --- Actualizar el registro histórico DESPUÉS de usar los valores ---
    # Incrementamos el contador de enfrentamientos para ambos pares (A, B) y (B, A)
    historial_enfrentamientos[player_A][player_B] += 1
    historial_enfrentamientos[player_B][player_A] += 1

    # Incrementamos el contador de victorias del jugador que ganó
    if player_A_won == 1:
        historial_victorias[player_A][player_B] += 1 # A ganó a B
    else: # playerA_won == 0, significa que playerB ganó a playerA
        historial_victorias[player_B][player_A] += 1 # B ganó a A

print("Características históricas generadas.")

# 5. Añadir las nuevas columnas al DataFrame ordenado
df_partidos['player_A_vs_B_matches'] = player_A_vs_B_matches_list
df_partidos['player_A_wins_vs_B'] = player_A_wins_vs_B_list
df_partidos['player_B_wins_vs_A'] = player_B_wins_vs_A_list

# --- VERSIÓN ROBUSTA (SIN DIVISIÓN) ---
print("Calculando features H2H finales...")

# Feature 1: Total de partidos (Tu 'Numero_enfrentamientos')
# (Es útil para que el modelo sepa si el H2H es fiable (ej. 10 partidos) o no (ej. 1 partido))
df_partidos["H2H_matches"] = df_partidos["player_A_vs_B_matches"]

# Feature 2: Ventaja H2H (La 'feature' potente)
# (Ej. 3-1 a favor de A -> +2. | 2-5 a favor de B -> -3. | 0-0 -> 0)
# Esta única feature reemplaza tus dos porcentajes y evita el NaN.
df_partidos["H2H_advantage_A"] = (
    df_partidos["player_A_wins_vs_B"] - df_partidos["player_B_wins_vs_A"]
)

# --- VERSIÓN ORIGINAL (Si aún la quieres) ---
# df_partidos["H2H_%_w_player_A"] = (
#     df_partidos["player_A_wins_vs_B"] / df_partidos["player_A_vs_B_matches"]
# )
# df_partidos["H2H_%_w_player_B"] = (
#     df_partidos["player_B_wins_vs_A"] / df_partidos["player_A_vs_B_matches"]
# )
# Rellenamos los NaN (0/0) con 0, lo cual es neutral
# df_partidos["H2H_%_w_player_A"] = df_partidos["H2H_%_w_player_A"].fillna(0)
# df_partidos["H2H_%_w_player_B"] = df_partidos["H2H_%_w_player_B"].fillna(0)

# Limpieza de columnas de trabajo
cols_a_borrar = [
    'player_A_vs_B_matches',
    'player_A_wins_vs_B',
    'player_B_wins_vs_A'
]
df_partidos = df_partidos.drop(columns=cols_a_borrar, errors='ignore')

print("¡Features H2H generadas con éxito!")
display(df_partidos[['player_A', 'player_B', 'H2H_matches', 'H2H_advantage_A']].head())

Procesando partidos cronológicamente para generar características históricas...
Características históricas generadas.
Calculando features H2H finales...
¡Features H2H generadas con éxito!


,player_A,player_B,H2H_matches,H2H_advantage_A
0,Ivo Heuberger,Cristiano Caratti,0,0
1,Younes El Aynaoui,Antony Dupuis,0,0
2,Fernando Vicente,Daniel Vacek,0,0
3,Fabrice Santoro,Rainer Schuettler,0,0
4,Younes El Aynaoui,Rainer Schuettler,0,0


In [ ]:
#Calcular victorias ultimos 10 partidos
# --- 2. Crear un DataFrame auxiliar para el historial de cada jugador ---
# Este DataFrame contendrá cada instancia de un jugador en un partido,
# junto con su 'indice' (como identificador de orden) y si ganó ese partido.

# Perspectiva de Jugador A
df_player_A_stats = df_partidos[['indice', 'player_A', 'playerA_won']].copy()
# Renombramos 'indice' a 'order_id' para generalizar el concepto de orden
df_player_A_stats.columns = ['order_id', 'player_id', 'won_match']

# Perspectiva de Jugador B
df_player_B_stats = df_partidos[['indice', 'player_B', 'playerA_won']].copy()
# Si playerA_won es 1, B perdió (0); si es 0, B ganó (1)
df_player_B_B_won = 1 - df_player_B_stats['playerA_won']
df_player_B_stats = df_player_B_stats.drop(columns=['playerA_won'])
df_player_B_stats.columns = ['order_id', 'player_id']
df_player_B_stats['won_match'] = df_player_B_B_won


# Unir ambos para tener un registro de cada partido que jugó CADA jugador
# Ordenar nuevamente para asegurar la secuencia correcta para las operaciones de ventana
# Primero por el identificador de orden ('order_id') y luego por el ID del jugador
df_all_player_matches = pd.concat([df_player_A_stats, df_player_B_stats]).sort_values(
    by=['order_id', 'player_id']
).reset_index(drop=True)

print("\n--- DataFrame Auxiliar de Todos los Partidos por Jugador (primeras 10 filas) ---")
print(df_all_player_matches.head(10))
print("-" * 50)

# --- 3. Calcular las victorias recientes y el conteo de partidos para cada jugador ---
WINDOW_SIZE = 10 # La ventana de los últimos 10 partidos

print(f"Calculando victorias y partidos en los últimos {WINDOW_SIZE} partidos previos por jugador...")

# Para cada jugador (agrupando por 'player_id'):
#   - Se desplaza `won_match` en 1 posición hacia abajo (`.shift(1)`) para excluir el partido actual.
#   - Se aplica una ventana móvil (`.rolling(window=WINDOW_SIZE, min_periods=1)`) para sumar las victorias.
#   - `min_periods=1` asegura que se calcule incluso si hay menos de 10 partidos previos.
#   - Se rellena cualquier `NaN` resultante con 0.

df_all_player_matches['wins_in_window'] = df_all_player_matches.groupby('player_id')['won_match'].transform(
    lambda x: x.shift(1).rolling(window=WINDOW_SIZE, min_periods=WINDOW_SIZE).sum()
).fillna(0).astype(int)

df_all_player_matches['matches_in_window'] = df_all_player_matches.groupby('player_id')['won_match'].transform(
    lambda x: x.shift(1).rolling(window=WINDOW_SIZE, min_periods=WINDOW_SIZE).count()
).fillna(0).astype(int)


# Calcular el porcentaje de victorias, manejando la división por cero
# np.where(condicion, valor_si_true, valor_si_false)
df_all_player_matches['win_percentage_in_window'] = np.where(
    df_all_player_matches['matches_in_window'] > 0,
    df_all_player_matches['wins_in_window'] / df_all_player_matches['matches_in_window'],
    0.0 # Si no hay partidos previos en la ventana, el porcentaje es 0.0
)

print("\n--- DataFrame Auxiliar con Victorias y Porcentajes Recientes Calculados (primeras 15 filas) ---")
print(df_all_player_matches.head(15))
print("-" * 50)

# --- 4. Unir los resultados al DataFrame original `df_partidos` ---

# Renomar columnas para la unión con playerA
playerA_recent_stats = df_all_player_matches[['order_id', 'player_id', 'wins_in_window', 'win_percentage_in_window']].rename(
    columns={
        'player_id': 'player_A',
        'wins_in_window': 'playerA_wins_last_10',
        'win_percentage_in_window': 'playerA_%_wins_last_10'
    }
)

# Renomar columnas para la unión con playerB
playerB_recent_stats = df_all_player_matches[['order_id', 'player_id', 'wins_in_window', 'win_percentage_in_window']].rename(
    columns={
        'player_id': 'player_B',
        'wins_in_window': 'playerB_wins_last_10',
        'win_percentage_in_window': 'playerB_%_wins_last_10'
    }
)

# Unir las estadísticas recientes de playerA al DataFrame original
# Usamos 'indice' para la unión, que corresponde a 'order_id' en el auxiliar
df_partidos = pd.merge(df_partidos, playerA_recent_stats,
                    left_on=['indice', 'player_A'],
                    right_on=['order_id', 'player_A'],
                    how='left')
df_partidos.drop(columns=['order_id'], inplace=True) # Eliminar columna auxiliar 'order_id' de la primera unión

# Unir las estadísticas recientes de playerB al DataFrame final
# Usamos 'indice' para la unión, que corresponde a 'order_id' en el auxiliar
df_partidos = pd.merge(df_partidos, playerB_recent_stats,
                    left_on=['indice', 'player_B'],
                    right_on=['order_id', 'player_B'],
                    how='left')
df_partidos.drop(columns=['order_id'], inplace=True) # Eliminar columna auxiliar 'order_id' de la segunda unión

# Rellenar cualquier NaN que pudiera quedar y asegurar los tipos de datos correctos.
df_partidos['playerA_wins_last_10'] = df_partidos['playerA_wins_last_10'].fillna(0).astype(int)
df_partidos['playerB_wins_last_10'] = df_partidos['playerB_wins_last_10'].fillna(0).astype(int)
df_partidos['playerA_%_wins_last_10'] = df_partidos['playerA_%_wins_last_10'].fillna(0.0)
df_partidos['playerB_%_wins_last_10'] = df_partidos['playerB_%_wins_last_10'].fillna(0.0)


--- DataFrame Auxiliar de Todos los Partidos por Jugador (primeras 10 filas) ---
   order_id          player_id  won_match
0         0  Cristiano Caratti          1
1         0      Ivo Heuberger          0
2         1      Antony Dupuis          0
3         1  Younes El Aynaoui          1
4         2       Daniel Vacek          1
5         2   Fernando Vicente          0
6         3    Fabrice Santoro          1
7         3  Rainer Schuettler          0
8         4  Rainer Schuettler          1
9         4  Younes El Aynaoui          0
--------------------------------------------------
Calculando victorias y partidos en los últimos 10 partidos previos por jugador...

--- DataFrame Auxiliar con Victorias y Porcentajes Recientes Calculados (primeras 15 filas) ---
    order_id          player_id  won_match  wins_in_window  matches_in_window  \
0          0  Cristiano Caratti          1               0                  0   
1          0      Ivo Heuberger          0               0      

In [ ]:
# Creación variable minutos acumulados en el torneo de ese año para cada jugador

from collections import defaultdict

# --- PASO 0: ¡LA LÍNEA DE SEGURIDAD! ---
# Asegura que el DataFrame esté en orden cronológico antes de iterar.
# Esto es vital para que 'cumulative' (acumulativo) funcione.
print("Asegurando el orden cronológico (por 'indice')...")
df_partidos = df_partidos.sort_values(by='indice', ascending=True)

# Paso 1: Crear una clave única para cada edición del torneo
df_partidos['tourney_year_id'] = df_partidos['Tournament'] + '_' + df_partidos['Date'].dt.year.astype(str)

# Inicializar diccionarios
player_cumulative_minutes = defaultdict(lambda: defaultdict(float))
player_matches_played = defaultdict(lambda: defaultdict(int))

# Inicializar listas para almacenar los resultados
playerA_cumulative_minutes_list = []
playerB_cumulative_minutes_list = []
playerA_matches_played_list = []
playerB_matches_played_list = []

print("Calculando minutos acumulados y partidos jugados por cada jugador en cada edición de torneo...")

# Iterar a través de cada fila del DataFrame (AHORA ORDENADO)
for index, row in df_partidos.iterrows():
    tourney_id = row['tourney_year_id']
    playerA_id = row['player_A']
    playerB_id = row['player_B']
    minutes = row['duracion_min']

    # Obtener los valores acumulados antes del partido actual
    current_playerA_minutes = player_cumulative_minutes[tourney_id][playerA_id]
    current_playerB_minutes = player_cumulative_minutes[tourney_id][playerB_id]
    current_playerA_matches = player_matches_played[tourney_id][playerA_id]
    current_playerB_matches = player_matches_played[tourney_id][playerB_id]

    # Añadir los valores a las listas
    playerA_cumulative_minutes_list.append(current_playerA_minutes)
    playerB_cumulative_minutes_list.append(current_playerB_minutes)
    playerA_matches_played_list.append(current_playerA_matches)
    playerB_matches_played_list.append(current_playerB_matches)

    # Actualizar los valores después del partido actual
    player_cumulative_minutes[tourney_id][playerA_id] += minutes
    player_cumulative_minutes[tourney_id][playerB_id] += minutes
    player_matches_played[tourney_id][playerA_id] += 1
    player_matches_played[tourney_id][playerB_id] += 1

print("Cálculo completado.")

# Añadir las nuevas columnas al DataFrame
df_partidos['playerA_cumulative_minutes'] = playerA_cumulative_minutes_list
df_partidos['playerB_cumulative_minutes'] = playerB_cumulative_minutes_list
df_partidos['playerA_matches_played_tourney'] = playerA_matches_played_list
df_partidos['playerB_matches_played_tourney'] = playerB_matches_played_list

# Opcional: Eliminar la columna auxiliar
df_partidos = df_partidos.drop(columns=['tourney_year_id'])

print("\n--- ¡Éxito! Features de fatiga de torneo creadas ---")
cols_a_mostrar = [
    'player_A', 'playerA_cumulative_minutes', 'playerA_matches_played_tourney',
    'player_B', 'playerB_cumulative_minutes', 'playerB_matches_played_tourney'
]
display(df_partidos[cols_a_mostrar].tail(5)) # .tail() es bueno para ver acumulaciones

Asegurando el orden cronológico (por 'indice')...
Calculando minutos acumulados y partidos jugados por cada jugador en cada edición de torneo...
Cálculo completado.

--- ¡Éxito! Features de fatiga de torneo creadas ---


,player_A,playerA_cumulative_minutes,playerA_matches_played_tourney,player_B,playerB_cumulative_minutes,playerB_matches_played_tourney
258296,David Jorda Sanchis,101.0,1,Ignacio Carou,0.0,0
258297,Guido Ivan Justo,78.0,1,Federico Aguilar Cardozo,0.0,0
258298,Nicolas Kicker,101.0,1,Christoph Negritu,0.0,0
258299,Shunsuke Mitsui,0.0,0,Ryan Seggerman,79.0,1
258300,Facundo Diaz Acosta,0.0,0,Marco Cecchinato,0.0,0


In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

def calculate_elo_by_surface(df, initial_elo=1500, k_base=20, k_gs=40, k_atp=30, k_challenger=15):
    """
    Calcula ELO con un K-Factor dinámico Y segregado por superficie.
    Asume 'Tier_Numeric' y 'Surface_Encoded' existen.
    Asume que el df está limpio (sin duplicados de 'indice') y
    que 'player_A'/'player_B' están limpios (.strip().lower()).
    """

    # --- 1. LÍNEA DE SEGURIDAD CRÍTICA ---
    # (Asegura orden cronológico)
    df = df.sort_values(by='indice', ascending=True)

    # --- 2. Extraer datos a arrays de NumPy (rápido) ---
    player_A_list = df['player_A'].values
    player_B_list = df['player_B'].values
    playerA_won_list = df['playerA_won'].values
    tier_list = df['Tier_Numeric'].values
    surface_list = df['Surface_Encoded'].values # Clave de segregación

    n_partidos = len(df)

    # --- 3. Diccionario ELO (Segregado) ---
    # Formato: {codigo_superficie: {jugador: elo}}
    player_elos_by_surface = defaultdict(lambda: defaultdict(lambda: initial_elo))

    elo_A_results = np.full(n_partidos, np.nan)
    elo_B_results = np.full(n_partidos, np.nan)

    print(f"Calculando ELO DINÁMICO POR SUPERFICIE para {n_partidos} partidos...")

    # Mapeo de K-Factors
    k_map = { 5: k_gs, 4: k_atp, 3: k_atp, 2: k_base, 1: k_challenger }

    # --- 4. Bucle Rápido ---
    for i in range(n_partidos):
        player_A = player_A_list[i]
        player_B = player_B_list[i]
        playerA_won = playerA_won_list[i]
        tier = tier_list[i]
        surface = surface_list[i] # El código de superficie (ej. 0, 1, 3...)

        k_factor = k_map.get(tier, k_base)

        # --- Obtener el diccionario ELO para ESTA superficie ---
        elo_dict_surface = player_elos_by_surface[surface]

        # (Ya no es necesario 'if player_A not in ...'
        #  porque defaultdict(lambda: initial_elo) lo maneja)
        elo_playerA = elo_dict_surface[player_A]
        elo_playerB = elo_dict_surface[player_B]

        # Guardar Elos (pre-partido) -> Sin fuga de datos
        elo_A_results[i] = elo_playerA
        elo_B_results[i] = elo_playerB

        # Lógica de actualización (estándar)
        if playerA_won == 1:
            winner_elo, loser_elo = elo_playerA, elo_playerB
            winner_name, loser_name = player_A, player_B
        else:
            winner_elo, loser_elo = elo_playerB, elo_playerA
            winner_name, loser_name = player_B, player_A

        expected_winner = 1 / (1 + 10**((loser_elo - winner_elo) / 400))

        new_elo_winner = winner_elo + k_factor * (1 - expected_winner)
        new_elo_loser = loser_elo + k_factor * (0 - (1 - expected_winner))

        # --- Actualizar el diccionario ELO de ESTA superficie ---
        elo_dict_surface[winner_name] = new_elo_winner
        elo_dict_surface[loser_name] = new_elo_loser

    print("Cálculo de ELO por Superficie completado.")

    # --- 5. Asignar resultados al DataFrame ---
    df['playerA_elo_surface'] = elo_A_results
    df['playerB_elo_surface'] = elo_B_results

    return df

# --- 1. EJECUTAR EL CÁLCULO DE ELO ---
# (Asegúrate de que 'Tier_Numeric' y 'Surface_Encoded' existen)
df_partidos = calculate_elo_by_surface(df_partidos)

# --- 2. CREAR LAS FEATURES DIFERENCIALES ---
# (Reemplazando las del Turno 72)
df_partidos['elo_diff_surface'] = df_partidos['playerA_elo_surface'] - df_partidos['playerB_elo_surface']
df_partidos['elo_ratio_surface'] = df_partidos['playerA_elo_surface'] / df_partidos['playerB_elo_surface']

print("\n--- ¡Éxito! ELO por Superficie añadido ---")
display(df_partidos[['player_A', 'player_B', 'Surface_Encoded', 'playerA_elo_surface', 'playerB_elo_surface', 'elo_diff_surface']].head())


Calculando ELO DINÁMICO POR SUPERFICIE para 258301 partidos...
Cálculo de ELO por Superficie completado.

--- ¡Éxito! ELO por Superficie añadido ---


,player_A,player_B,Surface_Encoded,playerA_elo_surface,playerB_elo_surface,elo_diff_surface
0,Ivo Heuberger,Cristiano Caratti,2,1500.0,1500.0,0.0
1,Younes El Aynaoui,Antony Dupuis,2,1500.0,1500.0,0.0
2,Fernando Vicente,Daniel Vacek,2,1500.0,1500.0,0.0
3,Fabrice Santoro,Rainer Schuettler,2,1500.0,1500.0,0.0
4,Younes El Aynaoui,Rainer Schuettler,2,1510.0,1490.0,20.0


In [ ]:
# Creación de otras variables
df_partidos["Diferencia_forma_10_partidos"]  = df_partidos['playerA_%_wins_last_10'] - df_partidos['playerB_%_wins_last_10']
df_partidos["Diferencia_minutos_acumulados"]  = df_partidos['playerA_cumulative_minutes'] - df_partidos['playerB_cumulative_minutes']
df_partidos["Diferencia_partidos_torneo"] = df_partidos['playerA_matches_played_tourney'] - df_partidos['playerB_matches_played_tourney']
df_partidos ["Primer_partido_torneo_playerA"] = df_partidos["playerA_matches_played_tourney"] == 0
df_partidos ["Primer_partido_torneo_playerB"] = df_partidos["playerB_matches_played_tourney"] == 0

# --- NUEVA FEATURE: LOGARITMO DEL RANKING ---
# Aplicamos logaritmo para linealizar la fuerza de los jugadores.
# Rellenamos 0 o NaNs con un ranking bajo (ej. 500) para evitar log(0) = -inf
def get_log_rank(series):
    return np.log(series.replace(0, np.nan).fillna(500))

print("Calculando Logaritmos del Ranking...")
df_partidos ['log_Rk_A'] = get_log_rank(df_partidos ['Rk'])
df_partidos ['log_Rk_B'] = get_log_rank(df_partidos ['vRk'])
df_partidos ['log_rank_diff'] = df_partidos ['log_Rk_A'] - df_partidos ['log_Rk_B']

#Elimino 'log_Rk_A' y 'log_Rk_B'

df_partidos  = df_partidos .drop(columns=['log_Rk_A', 'log_Rk_B'], errors='ignore')

Calculando Logaritmos del Ranking...


In [ ]:
# Media historica, ultimos partidos y diferencia en varias variables por superficie y tipo de torneo
import pandas as pd
import numpy as np
from collections import defaultdict

print("--- REINICIO PIPELINE EWM (v14 - EWM + HistAvg + Momentum) ---")

# --- PASO 0: Limpieza de columnas corruptas (si existen) ---
print("Paso 0: Eliminando columnas EWM/HistAvg antiguas (corruptas)...")
stats_genericas_v14 = ['DR', 'A%', 'DF%', '1st%', '2nd%', 'BPSvd%']
surface_codes_v14 = df_partidos['Surface_Encoded'].unique()
columnas_a_borrar = []
for role in ['playerA', 'playerB']:
    for stat in stats_genericas_v14:
        for code in surface_codes_v14:
            # Nombres de v12/v13
            columnas_a_borrar.append(f'{role}_form_prev_{stat}_{code}')
            # Nombres de v14 (EWM, HistAvg, Momentum)
            columnas_a_borrar.append(f'{role}_EWM_{stat}_{code}')
            columnas_a_borrar.append(f'{role}_HistAvg_{stat}_{code}')
            columnas_a_borrar.append(f'{role}_Momentum_{stat}_{code}')
df_partidos = df_partidos.drop(columns=columnas_a_borrar, errors='ignore')
print("Columnas antiguas eliminadas.")

# --- PASO 1: Asegurar limpieza estructural (del Turno 81) ---
print("Paso 1: Asegurando limpieza (duplicados, índice, nombres)...")
df_partidos = df_partidos.drop_duplicates(subset=['indice'], keep='first').reset_index(drop=True)
df_partidos['player_A'] = df_partidos['player_A'].astype(str).str.strip().str.lower()
df_partidos['player_B'] = df_partidos['player_B'].astype(str).str.strip().str.lower()
df_partidos['indice'] = df_partidos['indice'].astype(int)

# --- PASO 2: LÓGICA DE BUCLE (v14 - Lento pero Completo) ---
variables_a_promediar = ['DR', 'A%', 'DF%', '1st%', '2nd%', 'BPSvd%']
codigos_superficie_all = df_partidos['Surface_Encoded'].unique()
print(f"Superficies a segregar: {codigos_superficie_all}")

# --- Historiales para ambas métricas ---
# 1. EWM (Forma Reciente)
historial_ewm = defaultdict(lambda: defaultdict(float))
span = 10
alpha = 2 / (span + 1)
alpha_decay = 1 - alpha

# 2. Media Histórica (Largo Plazo)
historial_expanding = defaultdict(lambda: defaultdict(lambda: (0.0, 0))) # (suma, contador)

# Listas de resultados (para construir el DataFrame al final)
listas_resultados = defaultdict(list)

print("Iniciando bucle .iterrows() (Esto tardará 15-30 minutos)...")

# Iterar cronológicamente
for _, row in df_partidos.iterrows():

    pA = row['player_A']
    pB = row['player_B']
    surface_actual = row['Surface_Encoded']

    for player, role in [(pA, 'playerA'), (pB, 'playerB')]:
        for var in variables_a_promediar:
            # Obtener la stat de ESTE partido (o 0)
            stat_col = f'{role}_{var}'
            stat_partido = pd.to_numeric(row.get(stat_col), errors='coerce')
            if pd.isna(stat_partido): stat_partido = 0

            # Iterar sobre TODAS las superficies (para actualización y decaimiento)
            for s_code in codigos_superficie_all:

                # --- 1. CÁLCULO DE EWM (Forma Reciente) ---
                key_ewm = f'ewm_{var}_{s_code}'
                col_ewm = f'{role}_EWM_{var}_{s_code}'

                ewm_anterior = historial_ewm[player][key_ewm]
                listas_resultados[col_ewm].append(ewm_anterior) # Guardar EWM (pre-partido)

                input_stat_ewm = stat_partido if s_code == surface_actual else 0
                nuevo_ewm = (input_stat_ewm * alpha) + (ewm_anterior * alpha_decay)
                historial_ewm[player][key_ewm] = nuevo_ewm # Actualizar historial

                # --- 2. CÁLCULO DE MEDIA HISTÓRICA (Largo Plazo) ---
                key_hist = f'hist_{var}_{s_code}'
                col_hist = f'{role}_HistAvg_{var}_{s_code}'

                suma_anterior, contador_anterior = historial_expanding[player][key_hist]

                media_anterior = 0.0 if contador_anterior == 0 else (suma_anterior / contador_anterior)
                listas_resultados[col_hist].append(media_anterior) # Guardar HistAvg (pre-partido)

                if s_code == surface_actual:
                    nueva_suma = suma_anterior + stat_partido
                    nuevo_contador = contador_anterior + 1
                    historial_expanding[player][key_hist] = (nueva_suma, nuevo_contador) # Actualizar historial

print("Bucle completado. Asignando resultados...")

# --- PASO 3: Asignar resultados ---
for col_name, lista_valores in listas_resultados.items():
    if len(lista_valores) == len(df_partidos):
        df_partidos[col_name] = lista_valores
    else:
        print(f"¡Error de longitud! Col: {col_name}.")

# --- PASO 4: ¡Calcular el "Momentum" (Tu objetivo)! ---
print("Calculando diferencias (Momentum)...")
for role in ['playerA', 'playerB']:
    for var in variables_a_promediar:
        for s_code in codigos_superficie_all:
            col_ewm = f'{role}_EWM_{var}_{s_code}'
            col_hist = f'{role}_HistAvg_{var}_{s_code}'
            col_diff = f'{role}_Momentum_{var}_{s_code}' # (EWM - HistAvg)

            # Rellenar NaNs (por si acaso) antes de restar
            df_partidos[col_ewm] = df_partidos[col_ewm].fillna(0)
            df_partidos[col_hist] = df_partidos[col_hist].fillna(0)

            df_partidos[col_diff] = df_partidos[col_ewm] - df_partidos[col_hist]

print("\n--- ¡Éxito! Features de 'Momentum' (v14) añadidas ---")
display(df_partidos[[col for col in df_partidos.columns if 'playerA_Momentum_DR' in col]].head())

--- REINICIO PIPELINE EWM (v14 - EWM + HistAvg + Momentum) ---
Paso 0: Eliminando columnas EWM/HistAvg antiguas (corruptas)...
Columnas antiguas eliminadas.
Paso 1: Asegurando limpieza (duplicados, índice, nombres)...
Superficies a segregar: [2 4 3 1 0]
Iniciando bucle .iterrows() (Esto tardará 15-30 minutos)...
Bucle completado. Asignando resultados...


C:\Users\UJA\AppData\Local\Temp\ipykernel_2724\2123565587.py:99: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_partidos[col_name] = lista_valores
C:\Users\UJA\AppData\Local\Temp\ipykernel_2724\2123565587.py:99: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_partidos[col_name] = lista_valores
C:\Users\UJA\AppData\Local\Temp\ipykernel_2724\2123565587.py:99: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all col

Calculando diferencias (Momentum)...


C:\Users\UJA\AppData\Local\Temp\ipykernel_2724\2123565587.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_partidos[col_diff] = df_partidos[col_ewm] - df_partidos[col_hist]
C:\Users\UJA\AppData\Local\Temp\ipykernel_2724\2123565587.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_partidos[col_diff] = df_partidos[col_ewm] - df_partidos[col_hist]
C:\Users\UJA\AppData\Local\Temp\ipykernel_2724\2123565587.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` 


--- ¡Éxito! Features de 'Momentum' (v14) añadidas ---


C:\Users\UJA\AppData\Local\Temp\ipykernel_2724\2123565587.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_partidos[col_diff] = df_partidos[col_ewm] - df_partidos[col_hist]
C:\Users\UJA\AppData\Local\Temp\ipykernel_2724\2123565587.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_partidos[col_diff] = df_partidos[col_ewm] - df_partidos[col_hist]
C:\Users\UJA\AppData\Local\Temp\ipykernel_2724\2123565587.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` 

,playerA_Momentum_DR_2,playerA_Momentum_DR_4,playerA_Momentum_DR_3,playerA_Momentum_DR_1,playerA_Momentum_DR_0
0,0.000000,0.0,0.0,0.0,0.0
1,0.000000,0.0,0.0,0.0,0.0
2,0.000000,0.0,0.0,0.0,0.0
3,0.000000,0.0,0.0,0.0,0.0
4,-1.227273,0.0,0.0,0.0,0.0


In [ ]:
# CALCULAR EL PORCENTAJE DE VICTORIAS HISTÓRICO POR JUGADOR Y SUPERFICIE
import pandas as pd
import numpy as np
from collections import defaultdict

print("--- REINICIO PIPELINE (v16 - % Victorias Histórico y Reciente (Corregido)) ---")

# --- PASO 0: Limpieza de columnas corruptas (si existen) ---
print("Paso 0: Eliminando columnas EWM antiguas (corruptas)...")
surface_codes_v16 = df_partidos['Surface_Encoded'].unique()
columnas_a_borrar = []
for role in ['playerA', 'playerB']:
    for s_code in surface_codes_v16:
        columnas_a_borrar.append(f'{role}_WinPct_Last10_Surf{s_code}')
        columnas_a_borrar.append(f'{role}_HistWinPct_Surf{s_code}')
        columnas_a_borrar.append(f'{role}_Momentum_WinPct_Surf{s_code}')
df_partidos = df_partidos.drop(columns=columnas_a_borrar, errors='ignore')
print("Columnas antiguas eliminadas.")

# --- PASO 1: Asegurar limpieza estructural (del Turno 81) ---
print("Paso 1: Asegurando limpieza (duplicados, índice, nombres)...")
df_partidos = df_partidos.drop_duplicates(subset=['indice'], keep='first').reset_index(drop=True)
df_partidos['player_A'] = df_partidos['player_A'].astype(str).str.strip().str.lower()
df_partidos['player_B'] = df_partidos['player_B'].astype(str).str.strip().str.lower()
df_partidos['indice'] = df_partidos['indice'].astype(int)

# --- PASO 2: LÓGICA DE BUCLE (v16) ---
codigos_superficie_all = df_partidos['Surface_Encoded'].unique()
print(f"Superficies a segregar: {codigos_superficie_all}")

# --- Historiales para ambas métricas ---
# 1. Media Histórica (Largo Plazo)
historial_expanding = defaultdict(lambda: defaultdict(lambda: (0.0, 0))) # (suma_victorias, contador_partidos)
# 2. Media Reciente (Últimos 10)
historial_rolling = defaultdict(lambda: defaultdict(lambda: [])) # (lista de [0, 1, 1...])

# Listas de resultados (para construir el DataFrame al final)
listas_resultados = defaultdict(list)

print("Iniciando bucle .iterrows() (Esto tardará 15-30 minutos)...")

# Iterar cronológicamente
for _, row in df_partidos.iterrows():

    pA = row['player_A']
    pB = row['player_B']
    surface_actual = row['Surface_Encoded']

    # Determinar quién ganó
    pA_won = row['playerA_won']
    pB_won = 1 - pA_won

    for player, role, won in [(pA, 'playerA', pA_won), (pB, 'playerB', pB_won)]:

        # Iterar sobre TODAS las superficies (para que las listas de resultados crezcan uniformemente)
        for s_code in codigos_superficie_all:

            key_hist = f'Hist_{s_code}' # (ej. 'Hist_3')
            key_roll = f'Roll_{s_code}' # (ej. 'Roll_3')

            col_hist_avg = f'{role}_HistWinPct_Surf{s_code}'
            col_roll_avg = f'{role}_WinPct_Last10_Surf{s_code}'

            # --- 1. CÁLCULO DE MEDIA HISTÓRICA (Largo Plazo) ---
            suma_anterior, contador_anterior = historial_expanding[player][key_hist]
            media_anterior_hist = 0.0 if contador_anterior == 0 else (suma_anterior / contador_anterior)
            listas_resultados[col_hist_avg].append(media_anterior_hist) # Guardar HistAvg (pre-partido)

            # --- 2. CÁLCULO DE MEDIA RECIENTE (Últimos 10) ---
            lista_anterior_roll = historial_rolling[player][key_roll]
            media_anterior_roll = 0.0 if not lista_anterior_roll else (sum(lista_anterior_roll) / len(lista_anterior_roll))
            listas_resultados[col_roll_avg].append(media_anterior_roll) # Guardar Last10 (pre-partido)

            # --- 3. Actualizar historiales (SOLO si el partido fue en esta superficie) ---
            if s_code == surface_actual:
                # Actualizar Histórico
                nueva_suma = suma_anterior + won
                nuevo_contador = contador_anterior + 1
                historial_expanding[player][key_hist] = (nueva_suma, nuevo_contador)

                # Actualizar Reciente (Rolling)
                lista_anterior_roll.append(won)
                # Mantener solo los últimos 10
                historial_rolling[player][key_roll] = lista_anterior_roll[-10:]

print("Bucle completado. Asignando resultados...")

# --- PASO 3: Asignar resultados ---
for col_name, lista_valores in listas_resultados.items():
    if len(lista_valores) == len(df_partidos):
        df_partidos[col_name] = lista_valores
    else:
        print(f"¡Error de longitud! Col: {col_name}.")

# --- PASO 4: ¡Calcular el "Momentum" (Tu objetivo)! ---
print("Calculando diferencias (Momentum)...")
for role in ['playerA', 'playerB']:
    for s_code in codigos_superficie_all:
        col_recent = f'{role}_WinPct_Last10_Surf{s_code}'
        col_hist = f'{role}_HistWinPct_Surf{s_code}'
        col_diff = f'{role}_Momentum_WinPct_Surf{s_code}'

        df_partidos[col_diff] = df_partidos[col_recent] - df_partidos[col_hist]

print("\n--- ¡Éxito! Features de '% Victorias por Superficie' (v16) añadidas ---")
display(df_partidos[[col for col in df_partidos.columns if 'playerA_Momentum_WinPct' in col]].head())

--- REINICIO PIPELINE (v16 - % Victorias Histórico y Reciente (Corregido)) ---
Paso 0: Eliminando columnas EWM antiguas (corruptas)...
Columnas antiguas eliminadas.
Paso 1: Asegurando limpieza (duplicados, índice, nombres)...
Superficies a segregar: [2 4 3 1 0]
Iniciando bucle .iterrows() (Esto tardará 15-30 minutos)...
Bucle completado. Asignando resultados...
Calculando diferencias (Momentum)...

--- ¡Éxito! Features de '% Victorias por Superficie' (v16) añadidas ---


,playerA_Momentum_WinPct_Surf2,playerA_Momentum_WinPct_Surf4,playerA_Momentum_WinPct_Surf3,playerA_Momentum_WinPct_Surf1,playerA_Momentum_WinPct_Surf0
0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0


In [ ]:
#Creo variables nuevas con el fin de eliminar variables con nombre playerA o playerB

import pandas as pd
import numpy as np

print("--- PASO 1: Creando Prerrequisitos (Pipeline v7.Final - Fatiga EWM) ---")

# --- 1. Limpieza de columnas corruptas (si existen) ---
print("Limpiando columnas de Fatiga EWM antiguas...")
columnas_ewm_fatiga_a_borrar = [
    'avg_juegos_bo3_A', 'avg_juegos_bo5_A', 'avg_sets_bo3_A', 'avg_sets_bo5_A',
    'avg_juegos_bo3_B', 'avg_juegos_bo5_B', 'avg_sets_bo3_B', 'avg_sets_bo5_B',
    'Juegos_Bo3', 'Juegos_Bo5', 'Sets_Bo3', 'Sets_Bo5' # Temporales
]
df_partidos = df_partidos.drop(columns=columnas_ewm_fatiga_a_borrar, errors='ignore')

# --- 2. Asegurar limpieza estructural (del Turno 81) ---
print("Asegurando limpieza (duplicados, índice, nombres)...")
df_partidos = df_partidos.drop_duplicates(subset=['indice'], keep='first').reset_index(drop=True)
df_partidos['player_A'] = df_partidos['player_A'].astype(str).str.strip().str.lower()
df_partidos['player_B'] = df_partidos['player_B'].astype(str).str.strip().str.lower()
df_partidos['indice'] = df_partidos['indice'].astype(int)

# --- 3. Crear columnas segregadas ---
print("Creando columnas segregadas (Juegos_Bo3, Juegos_Bo5)...")
df_partidos['Juegos_Bo3'] = np.where(df_partidos['Best_of_5'] == 0, df_partidos['Total_juegos'], 0)
df_partidos['Juegos_Bo5'] = np.where(df_partidos['Best_of_5'] == 1, df_partidos['Total_juegos'], 0)
df_partidos['Sets_Bo3'] = np.where(df_partidos['Best_of_5'] == 0, df_partidos['Numero_sets'], 0)
df_partidos['Sets_Bo5'] = np.where(df_partidos['Best_of_5'] == 1, df_partidos['Numero_sets'], 0)

# --- 4. Unstack (Método Rápido) ---
print("Desapilando (Unstack) DataFrame...")
df_temp = pd.DataFrame({
    'indice': pd.concat([df_partidos['indice'], df_partidos['indice']]),
    'Player': pd.concat([df_partidos['player_A'], df_partidos['player_B']]),
    'Juegos_Bo3': pd.concat([df_partidos['Juegos_Bo3'], df_partidos['Juegos_Bo3']]),
    'Juegos_Bo5': pd.concat([df_partidos['Juegos_Bo5'], df_partidos['Juegos_Bo5']]),
    'Sets_Bo3': pd.concat([df_partidos['Sets_Bo3'], df_partidos['Sets_Bo3']]),
    'Sets_Bo5': pd.concat([df_partidos['Sets_Bo5'], df_partidos['Sets_Bo5']])
})
df_temp = df_temp.sort_values(by=['Player', 'indice'])

# --- 5. Calcular EWM ---
print("Calculando EWM (Fatiga)...")
span = 10
columnas_fatiga = ['Juegos_Bo3', 'Juegos_Bo5', 'Sets_Bo3', 'Sets_Bo5']
features_creadas_fatiga = []
ewm_lista = []

for col in columnas_fatiga:
    form_col_name = f'form_prev_{col}'
    features_creadas_fatiga.append(form_col_name)

    ewm_series = df_temp.groupby('Player')[col].ewm(
        span=span, min_periods=1, adjust=False
    ).mean().reset_index(level=0, drop=True)

    form_prev_series = ewm_series.groupby(df_temp['Player']).shift(1)
    ewm_lista.append(
        pd.DataFrame(form_prev_series.values, index=df_temp.index, columns=[form_col_name])
    )

df_temp = pd.concat([df_temp, *ewm_lista], axis=1)
df_temp.fillna(0, inplace=True)

# --- 6. Merge (Unión) ---
print("Uniendo (merge) resultados de Fatiga...")
cols_forma = ['indice', 'Player'] + features_creadas_fatiga
df_temp_forma = df_temp[cols_forma].copy()
df_temp_forma['indice'] = df_temp_forma['indice'].astype(int)
df_temp_forma['Player'] = df_temp_forma['Player'].astype(str)

# Unir para Player A
df_partidos = pd.merge(
    df_partidos, df_temp_forma,
    left_on=['indice', 'player_A'], right_on=['indice', 'Player'], how='left'
)
rename_map_A = {f: f'avg_{col.lower().replace("form_prev_","")}_A' for f, col in zip(features_creadas_fatiga, columnas_fatiga)}
df_partidos = df_partidos.rename(columns=rename_map_A).drop(columns=['Player', 'Player_y'], errors='ignore')

# Unir para Player B
df_partidos = pd.merge(
    df_partidos, df_temp_forma,
    left_on=['indice', 'player_B'], right_on=['indice', 'Player'], how='left'
)
rename_map_B = {f: f'avg_{col.lower().replace("form_prev_","")}_B' for f, col in zip(features_creadas_fatiga, columnas_fatiga)}
df_partidos = df_partidos.rename(columns=rename_map_B).drop(columns=['Player', 'Player_y'], errors='ignore')

# Limpieza final
columnas_finales_fatiga = list(rename_map_A.values()) + list(rename_map_B.values())
df_partidos[columnas_finales_fatiga] = df_partidos[columnas_finales_fatiga].fillna(0)
df_partidos = df_partidos.drop(columns=['Juegos_Bo3', 'Juegos_Bo5', 'Sets_Bo3', 'Sets_Bo5'], errors='ignore')

print("--- ¡Éxito! Prerrequisitos (v7.Final) creados. ---")

--- PASO 1: Creando Prerrequisitos (Pipeline v7.Final - Fatiga EWM) ---
Limpiando columnas de Fatiga EWM antiguas...
Asegurando limpieza (duplicados, índice, nombres)...
Creando columnas segregadas (Juegos_Bo3, Juegos_Bo5)...
Desapilando (Unstack) DataFrame...
Calculando EWM (Fatiga)...
Uniendo (merge) resultados de Fatiga...
--- ¡Éxito! Prerrequisitos (v7.Final) creados. ---


In [ ]:
import pandas as pd
import numpy as np

print("\n--- PASO 2: CREANDO FEATURES DIFERENCIALES FINALES (v17) ---")

# --- 1. Variables Simples ---
print("Creando features diferenciales (Simples)...")

df_partidos["Primer_partido_torneo_playerA"] = df_partidos["playerA_matches_played_tourney"] == 0
df_partidos["Primer_partido_torneo_playerB"] = df_partidos["playerB_matches_played_tourney"] == 0
df_partidos ["Primer_partido_ambos"] = df_partidos["Primer_partido_torneo_playerA"] & df_partidos["Primer_partido_torneo_playerB"]

# --- 2. Diferenciales de Fatiga (Segregados Bo3/Bo5) ---
# (Esta línea ahora funcionará gracias al Paso 1)
print("Creando features diferenciales (Fatiga Bo3/Bo5)...")
df_partidos['diff_avg_juegos_bo3'] = df_partidos['avg_juegos_bo3_A'] - df_partidos['avg_juegos_bo3_B']
df_partidos['diff_avg_juegos_bo5'] = df_partidos['avg_juegos_bo5_A'] - df_partidos['avg_juegos_bo5_B']
df_partidos['diff_avg_sets_bo3'] = df_partidos['avg_sets_bo3_A'] - df_partidos['avg_sets_bo3_B']
df_partidos['diff_avg_sets_bo5'] = df_partidos['avg_sets_bo5_A'] - df_partidos['avg_sets_bo5_B']


# --- 3. Diferenciales de Stats (v14) (Segregados por Superficie) ---
print("Creando features diferenciales (Stats v14)...")

stats_genericas_v14 = ['DR', 'A%', 'DF%', '1st%', '2nd%', 'BPSvd%']
codigos_superficie_all = df_partidos['Surface_Encoded'].unique()

for var in stats_genericas_v14:
    for s_code in codigos_superficie_all:

        col_A_EWM = f'playerA_EWM_{var}_{s_code}'
        col_B_EWM = f'playerB_EWM_{var}_{s_code}'
        if col_A_EWM in df_partidos.columns and col_B_EWM in df_partidos.columns:
            df_partidos[f'diff_EWM_{var}_{s_code}'] = df_partidos[col_A_EWM] - df_partidos[col_B_EWM]

        col_A_Hist = f'playerA_HistAvg_{var}_{s_code}'
        col_B_Hist = f'playerB_HistAvg_{var}_{s_code}'
        if col_A_Hist in df_partidos.columns and col_B_Hist in df_partidos.columns:
            df_partidos[f'diff_HistAvg_{var}_{s_code}'] = df_partidos[col_A_Hist] - df_partidos[col_B_Hist]

        col_A_Mom = f'playerA_Momentum_{var}_{s_code}'
        col_B_Mom = f'playerB_Momentum_{var}_{s_code}'
        if col_A_Mom in df_partidos.columns and col_B_Mom in df_partidos.columns:
            df_partidos[f'diff_Momentum_{var}_{s_code}'] = df_partidos[col_A_Mom] - df_partidos[col_B_Mom]

# --- 4. Diferenciales de % Victorias (v16) (Segregados por Superficie) ---
print("Creando features diferenciales (% Victorias v16)...")

for s_code in codigos_superficie_all:

    col_A_WinPct10 = f'playerA_WinPct_Last10_Surf{s_code}'
    col_B_WinPct10 = f'playerB_WinPct_Last10_Surf{s_code}'
    if col_A_WinPct10 in df_partidos.columns and col_B_WinPct10 in df_partidos.columns:
        df_partidos[f'diff_WinPct_Last10_Surf{s_code}'] = df_partidos[col_A_WinPct10] - df_partidos[col_B_WinPct10]

    col_A_HistWinPct = f'playerA_HistWinPct_Surf{s_code}'
    col_B_HistWinPct = f'playerB_HistWinPct_Surf{s_code}'
    if col_A_HistWinPct in df_partidos.columns and col_B_HistWinPct in df_partidos.columns:
        df_partidos[f'diff_HistWinPct_Surf{s_code}'] = df_partidos[col_A_HistWinPct] - df_partidos[col_B_HistWinPct]

    col_A_MomWinPct = f'playerA_Momentum_WinPct_Surf{s_code}'
    col_B_MomWinPct = f'playerB_Momentum_WinPct_Surf{s_code}'
    if col_A_MomWinPct in df_partidos.columns and col_B_MomWinPct in df_partidos.columns:
        df_partidos[f'diff_Momentum_WinPct_Surf{s_code}'] = df_partidos[col_A_MomWinPct] - df_partidos[col_B_MomWinPct]

print("\n--- ¡Éxito! Todas las features diferenciales (v17) han sido creadas ---")
display(df_partidos[['indice', 'diff_avg_juegos_bo3', 'diff_EWM_DR_3']].head())


--- PASO 2: CREANDO FEATURES DIFERENCIALES FINALES (v17) ---
Creando features diferenciales (Simples)...
Creando features diferenciales (Fatiga Bo3/Bo5)...
Creando features diferenciales (Stats v14)...
Creando features diferenciales (% Victorias v16)...

--- ¡Éxito! Todas las features diferenciales (v17) han sido creadas ---


C:\Users\UJA\AppData\Local\Temp\ipykernel_2724\2800169108.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_partidos[f'diff_EWM_{var}_{s_code}'] = df_partidos[col_A_EWM] - df_partidos[col_B_EWM]
C:\Users\UJA\AppData\Local\Temp\ipykernel_2724\2800169108.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_partidos[f'diff_HistAvg_{var}_{s_code}'] = df_partidos[col_A_Hist] - df_partidos[col_B_Hist]
C:\Users\UJA\AppData\Local\Temp\ipykernel_2724\2800169108.py:44: PerformanceWarning: DataFrame is highly fragmented.  This is 

,indice,diff_avg_juegos_bo3,diff_EWM_DR_3
0,0,0.0,0.0
1,1,0.0,0.0
2,2,0.0,0.0
3,3,0.0,0.0
4,4,0.0,0.0


In [ ]:
# # #Elimino variables ya no necesarias

import pandas as pd
import numpy as np

# --- PASO 1: Desfragmentar el DataFrame (Soluciona el PerformanceWarning) ---
print(f"Columnas antes de la desfragmentación: {len(df_partidos.columns)}")
print("Desfragmentando el DataFrame (solucionando el PerformanceWarning)...")
# Esto fuerza a pandas a reasignar la memoria en un bloque contiguo
df_partidos = df_partidos.copy()
print("¡DataFrame desfragmentado!")


# --- PASO 2: Limpieza Final (Script del Turno 122) ---
print("\nIniciando la limpieza final de columnas...")

# A. Columnas de IDENTIFICACIÓN y OBJETIVO (Target)
columnas_a_conservar = [
    'indice',
    'Date',
    'Tournament',
    'player_A',
    'player_B',
    'Score',
    'playerA_won', # <- ¡El Target!
    'Total_juegos',  # (Target secundario)
    'Numero_sets',   # (Target secundario)
    'duracion_min',  # (Target secundario)
]

# B. FEATURES BASE (Contexto del partido)
features_base = [
    'Tier_Numeric',
    'Best_of_5',
    'Rd_Ordinal',
    'Surface_Encoded'
]
columnas_a_conservar.extend(features_base)

# C. FEATURES DIFERENCIALES (A vs B)
# (Todo lo demás será eliminado)
prefixes_diferenciales = [
    'ht_diff',
    "ht_sum",
    'age_diff',
    'hand_diff',
    'rank_diff',
    'rank_ratio',
    'H2H_matches',
    'H2H_advantage_A',
    'elo_diff_surface',
    'elo_ratio_surface',
    'Diferencia_forma_10_partidos',
    'Diferencia_minutos_acumulados',
    'Diferencia_partidos_torneo',
    'Primer_partido_ambos',
    'diff_avg_juegos',  # (Fatiga v7)
    'diff_avg_sets',    # (Fatiga v7)
    'diff_EWM',         # (Stats v14)
    'diff_HistAvg',     # (Stats v14)
    'diff_Momentum',    # (Stats v14)
    'diff_WinPct',      # (% Victorias v16)
    'diff_HistWinPct',  # (% Victorias v16)
    'diff_Momentum_WinPct' # (% Victorias v16, creado en v17)
]

# Añadir todas las columnas que coincidan con esos prefijos
all_columns = df_partidos.columns
for prefix in prefixes_diferenciales:
    columnas_a_conservar.extend(all_columns[all_columns.str.startswith(prefix)])

# Eliminar duplicados de la lista
columnas_a_conservar = sorted(list(set(columnas_a_conservar)))

# --- 3. Ejecutar la Limpieza ---
df_partidos = df_partidos[columnas_a_conservar]

print(f"\nColumnas DESPUÉS de la limpieza: {len(df_partidos.columns)}")
print("¡Limpieza completada! df_partidos está listo para el entrenamiento.")
display(df_partidos.head())

Columnas antes de la desfragmentación: 388
Desfragmentando el DataFrame (solucionando el PerformanceWarning)...
¡DataFrame desfragmentado!

Iniciando la limpieza final de columnas...

Columnas DESPUÉS de la limpieza: 137
¡Limpieza completada! df_partidos está listo para el entrenamiento.


,Best_of_5,Date,Diferencia_forma_10_partidos,Diferencia_minutos_acumulados,Diferencia_partidos_torneo,H2H_advantage_A,H2H_matches,Numero_sets,Primer_partido_ambos,Rd_Ordinal,...,elo_ratio_surface,hand_diff,ht_diff,ht_sum,indice,playerA_won,player_A,player_B,rank_diff,rank_ratio
0,3,2000-01-03,0.0,0.0,0,0,0,4.0,True,3.0,...,1.000000,0.0,10.0,366.0,0,0,ivo heuberger,cristiano caratti,8.0,1.03271
1,3,2000-01-03,0.0,0.0,0,0,0,4.0,True,3.0,...,1.000000,0.0,8.0,378.0,1,1,younes el aynaoui,antony dupuis,-50.0,0.392857
2,3,2000-01-03,0.0,0.0,0,0,0,4.0,True,3.0,...,1.000000,0.0,-10.0,370.0,2,0,fernando vicente,daniel vacek,6.0,1.113636
3,3,2000-01-03,0.0,0.0,0,0,0,4.0,True,7.0,...,1.000000,0.0,-2.0,358.0,3,1,fabrice santoro,rainer schuettler,-14.0,0.693878
4,3,2000-01-03,0.0,2.0,0,0,0,4.0,False,6.0,...,1.013423,0.0,13.0,373.0,4,0,younes el aynaoui,rainer schuettler,-15.0,0.673469


In [ ]:
import pandas as pd

# 1. Contar los NaNs en todas las columnas
nan_counts = df_partidos.isnull().sum()

# 2. Filtrar para mostrar solo las columnas que SÍ tienen NaNs
nan_counts_filtered = nan_counts[nan_counts > 0]

# 3. Ordenar para ver los peores problemas primero
nan_counts_sorted = nan_counts_filtered.sort_values(ascending=False)

if nan_counts_sorted.empty:
    print("¡Éxito! No se encontraron valores nulos (NaN) en ninguna columna.")
else:
    print("--- Columnas con Valores Nulos (NaN) Encontrados ---")
    print(nan_counts_sorted)

# (Opcional) Ver el total de celdas vacías en todo el DataFrame
total_nans = df_partidos.isnull().sum().sum()
print(f"\nTotal de celdas NaNs en el DataFrame: {total_nans}")

--- Columnas con Valores Nulos (NaN) Encontrados ---
duracion_min                     70125
Diferencia_minutos_acumulados    50174
ht_diff                          37235
ht_sum                           37235
age_diff                          1317
Score                               18
Numero_sets                         18
dtype: int64

Total de celdas NaNs en el DataFrame: 196122


In [ ]:
# Imputar NaN
import pandas as pd
import numpy as np

print("--- INICIANDO IMPUTACIÓN INTELIGENTE (Duración por Juegos) ---")

# 1. Eliminar filas corruptas (Score/Sets nulos) - Son insignificantes
df_partidos = df_partidos.dropna(subset=['Score', 'Numero_sets', 'Total_juegos'])
print(f"Filas después de limpieza básica: {len(df_partidos)}")

# --- 2. IMPUTACIÓN DE DURACIÓN (Regla: 1.7 min/juego) ---
# Calculamos cuántos faltan
n_duracion_faltante = df_partidos['duracion_min'].isnull().sum()
print(f"\nImputando {n_duracion_faltante} duraciones usando 'Total_juegos * 1.7'...")

# Aplicar fórmula solo donde es NaN
mask_duracion_nan = df_partidos['duracion_min'].isnull()
df_partidos.loc[mask_duracion_nan, 'duracion_min'] = df_partidos.loc[mask_duracion_nan, 'Total_juegos'] * 1.7

# Convertir a entero (opcional, por estética)
df_partidos['duracion_min'] = df_partidos['duracion_min'].astype(int)


# --- 3. IMPUTACIÓN DE FATIGA ACUMULADA ---
# Como la fatiga acumulada depende del pasado, y no podemos recalcular todo el historial ahora,
# rellenamos los huecos con 0 (asumimos neutralidad si no hay datos históricos).
print("Imputando 'Diferencia_minutos_acumulados' con 0 (Neutralidad)...")
df_partidos['Diferencia_minutos_acumulados'] = df_partidos['Diferencia_minutos_acumulados'].fillna(0)


# --- 4. IMPUTACIÓN DE ALTURA (Con Flag) ---
print("Procesando 'ht_diff' (Altura)...")
# 1. Crear la bandera (Le dice al modelo: "Este dato es falso/imputado")
df_partidos['ht_diff_is_missing'] = df_partidos['ht_diff'].isnull().astype(int)
df_partidos['ht_sum_is_missing'] = df_partidos['ht_sum'].isnull().astype(int)
# 2. Rellenar con 0 (Asumir que miden lo mismo es la apuesta más segura)
df_partidos['ht_diff'] = df_partidos['ht_diff'].fillna(0)
df_partidos['ht_sum'] = df_partidos['ht_sum'].fillna(0)

# --- 5. IMPUTACIÓN DE EDAD (Con Flag) ---
print("Procesando 'age_diff' (Edad)...")
df_partidos['age_diff_is_missing'] = df_partidos['age_diff'].isnull().astype(int)
df_partidos['age_diff'] = df_partidos['age_diff'].fillna(0)


# --- VERIFICACIÓN FINAL ---
print("\n--- Verificación Final de Nulos ---")
nulos_restantes = df_partidos.isnull().sum().sum()
if nulos_restantes == 0:
    print("¡ÉXITO TOTAL! El DataFrame está limpio y listo para el Feature Engineering.")
else:
    print(f"¡Aviso! Aún quedan {nulos_restantes} nulos (revisar si son columnas importantes).")
    # Mostrar qué columnas fallan
    print(df_partidos.isnull().sum()[df_partidos.isnull().sum() > 0])

display(df_partidos[['duracion_min', 'ht_diff', 'ht_diff_is_missing']].head())

--- INICIANDO IMPUTACIÓN INTELIGENTE (Duración por Juegos) ---
Filas después de limpieza básica: 257275

Imputando 70107 duraciones usando 'Total_juegos * 1.7'...
Imputando 'Diferencia_minutos_acumulados' con 0 (Neutralidad)...
Procesando 'ht_diff' (Altura)...
Procesando 'age_diff' (Edad)...

--- Verificación Final de Nulos ---
¡ÉXITO TOTAL! El DataFrame está limpio y listo para el Feature Engineering.


,duracion_min,ht_diff,ht_diff_is_missing
0,143,10.0,0
1,120,8.0,0
2,101,-10.0,0
3,118,-2.0,0
4,155,13.0,0


In [ ]:
#Elimino el año 2001 y 2000

df_partidos = df_partidos[df_partidos.Date.dt.year != 2000]
df_partidos = df_partidos[df_partidos.Date.dt.year != 2001]

